In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/sample_submission.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/item_meta.jsonl
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/test.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train_reviews.jsonl
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/README.md
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/example_popularity.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/experiments_template.csv
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/teamkit.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/tests/test_teamkit.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/validation/manifest.jso

In [2]:
from pathlib import Path

input_dir = Path("/kaggle/input")
files = sorted(
    path for path in input_dir.rglob("*")
    if path.is_file() and not path.name.startswith(".")
)

if not files:
    print("Add Input에서 대회 데이터와 팀 공통 검증 데이터를 연결해 주세요.")
else:
    for path in files:
        print(path)

/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/item_meta.jsonl
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/sample_submission.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/test.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train.csv
/kaggle/input/competitions/ssu-data-science-amazon-top-k-recommendation/train_reviews.jsonl
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/README.md
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/example_popularity.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/experiments_template.csv
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/teamkit.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/tests/test_teamkit.py
/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit/validation/manifest.jso

In [3]:
#셀 1. 팀 공통 검증 데이터 로딩
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

TEAMKIT_DIR = Path(
    "/kaggle/input/datasets/kimeoryeong/amazon-topk-teamkit/amazon_topk_teamkit"
)
VAL_DIR = TEAMKIT_DIR / "validation"

with (VAL_DIR / "manifest.json").open(encoding="utf-8") as file:
    manifest = json.load(file)

train = pd.read_csv(VAL_DIR / "val_train.csv", dtype=str)
truth = pd.read_csv(VAL_DIR / "val_truth.csv", dtype=str)
users = pd.read_csv(VAL_DIR / "val_users.csv", dtype=str)

print("manifest:", manifest)
for name, frame in [("train", train), ("truth", truth), ("users", users)]:
    print(f"{name}: {len(frame):,}행 | 열 이름: {frame.columns.tolist()}")

# 실제 열 이름이 다르면 위 출력에 맞춰 이 두 줄을 수정하세요.
USER_COL = "user_id"
ITEM_COL = "item_id"

for name, frame, columns in [
    ("train", train, [USER_COL, ITEM_COL]),
    ("truth", truth, [USER_COL, ITEM_COL]),
    ("users", users, [USER_COL]),
]:
    missing = set(columns) - set(frame.columns)
    if missing:
        raise ValueError(f"{name} 파일의 열 이름을 확인하세요. 필요한 열: {missing}")
    if frame[columns].isna().any().any():
        raise ValueError(f"{name} 파일에 빈 ID가 있어요.")

assert "f0dfad2f4431a876" in json.dumps(manifest), "공통 split ID를 확인하세요."
assert len(train) == 478_901, "검증 학습 데이터 행 수를 확인하세요."
assert len(users) == 36_043 and users[USER_COL].is_unique, "평가 사용자 목록을 확인하세요."
assert len(truth) == 137_041, "검증 정답 데이터 행 수를 확인하세요."
assert not truth.duplicated([USER_COL, ITEM_COL]).any(), "중복 정답 행이 있어요."

eval_user_ids = pd.Index(users[USER_COL])
item_ids = pd.Index(sorted(train[ITEM_COL].unique()))

train_rows = eval_user_ids.get_indexer(train[USER_COL])
train_cols = item_ids.get_indexer(train[ITEM_COL])
assert (train_rows >= 0).all(), "평가 사용자 목록에 없는 학습 사용자 ID가 있어요."

X = sparse.csr_matrix(
    (np.ones(len(train), dtype=np.float32), (train_rows, train_cols)),
    shape=(len(eval_user_ids), len(item_ids)),
)
X.data.fill(1.0)
X.sort_indices()

truth_pairs = pd.MultiIndex.from_frame(truth[[USER_COL, ITEM_COL]])
assert truth[USER_COL].isin(eval_user_ids).all(), "평가 사용자 목록에 없는 정답이 있어요."

print(f"학습 행렬: 사용자 {X.shape[0]:,}명 × 상품 {X.shape[1]:,}개")
print(f"검증 정답: {len(truth_pairs):,}건")

manifest: {'split_id': 'f0dfad2f4431a876', 'algorithm': 'per-user blake2b-128 rank over seed, user_id, item_id', 'seed': 42, 'holdout_fraction': 0.2, 'holdout_count': 'min(20, n-1, max(1, ceil(fraction*n))) for n>=2', 'source_train_sha256': '183de854f15b440ebbdf332dfa5c4b9a13812009fc637a043e09dc18f378d800', 'source_rows': 615942, 'source_users': 36043, 'fit_rows': 478901, 'truth_rows': 137041, 'validation_users': 36043, 'files_sha256': {'val_train.csv': '60e80f9e400d88057c1da9e2eaa9adda1594b31552cd102af07ca1ec8b5010c8', 'val_truth.csv': '228401f39c95a87282245462fc462ce6bff5f6ea6d6380da93c880584b62d900', 'val_users.csv': 'b31eb7a3555b0b77f53aa81f05e538cce02dac59fa0cea3f3d76b56ec45c816d'}}
train: 478,901행 | 열 이름: ['user_id', 'item_id', 'rating', 'timestamp']
truth: 137,041행 | 열 이름: ['user_id', 'item_id']
users: 36,043행 | 열 이름: ['user_id']
학습 행렬: 사용자 36,043명 × 상품 19,547개
검증 정답: 137,041건


In [4]:
#셀 2. Popularity + BM25-Shrink 재현 실험
from time import perf_counter

TOP_K = 20
NEIGHBORS = 2_000
K1, B = 1.2, 0.75
SHRINK, POWER = 50.0, 0.5
POP_WEIGHT, BM25_WEIGHT = 0.35, 0.65

started = perf_counter()

# 구현 가정 1: 상품을 문서, 사용자를 용어로 보고 BM25 가중치를 계산합니다.
item_user = X.T.tocsr()
item_lengths = np.diff(item_user.indptr).astype(np.float32)
user_freq = np.diff(X.indptr).astype(np.float32)
avg_length = item_lengths.mean()

idf = np.log1p(
    (X.shape[1] - user_freq + 0.5) / (user_freq + 0.5)
).astype(np.float32)
length_norm = K1 * (1.0 - B + B * item_lengths / avg_length)

weighted = item_user.copy()
weighted.data = (
    idf[weighted.indices]
    * np.repeat((K1 + 1.0) / (1.0 + length_norm), np.diff(weighted.indptr))
).astype(np.float32)

# 구현 가정 2: weighted dot / ((norm_i * norm_j) ** POWER + SHRINK)
norms = np.sqrt(np.asarray(weighted.multiply(weighted).sum(axis=1)).ravel())
weighted_t = weighted.T.tocsc()
neighbor_blocks = []

for start in range(0, X.shape[1], 128):
    end = min(start + 128, X.shape[1])
    block = (weighted[start:end] @ weighted_t).tocsr()

    indices = []
    values = []
    indptr = [0]

    for row in range(end - start):
        left, right = block.indptr[row:row + 2]
        cols = block.indices[left:right]
        dots = block.data[left:right]
        item = start + row

        mask = cols != item
        cols = cols[mask]
        dots = dots[mask]
        scores = dots / ((norms[item] * norms[cols]) ** POWER + SHRINK)

        # 점수 내림차순, 동점이면 상품 ID 오름차순으로 선택합니다.
        order = np.lexsort((cols, -scores))[:NEIGHBORS]
        indices.append(cols[order])
        values.append(scores[order])
        indptr.append(indptr[-1] + len(order))

    neighbor_blocks.append(sparse.csr_matrix(
        (np.concatenate(values), np.concatenate(indices), indptr),
        shape=(end - start, X.shape[1]),
    ))

W = sparse.vstack(neighbor_blocks, format="csr")
del neighbor_blocks
W.sort_indices()
print(f"BM25 이웃 계산 완료 | {perf_counter() - started:.1f}초", flush=True)

popularity = np.asarray(X.sum(axis=0)).ravel()
pop_order = np.lexsort((np.arange(X.shape[1]), -popularity))
pop_scores = popularity / popularity.max()

pop_top = np.empty((X.shape[0], TOP_K), dtype=np.int32)
fused_top = np.empty_like(pop_top)

for start in range(0, X.shape[0], 128):
    end = min(start + 128, X.shape[0])
    scores = (X[start:end] @ W).toarray()

    # 학습에서 이미 본 상품을 후보에서 제외합니다.
    seen_rows, seen_cols = X[start:end].nonzero()
    scores[seen_rows, seen_cols] = 0.0

    # 구현 가정 3: BM25는 사용자별 최대값, Pop은 전체 최대값으로 나눕니다.
    maxima = scores.max(axis=1, keepdims=True)
    np.divide(scores, maxima, out=scores, where=maxima > 0)
    scores *= BM25_WEIGHT
    scores += POP_WEIGHT * pop_scores
    scores[seen_rows, seen_cols] = -np.inf

    # 상품 열이 ID 순으로 정렬되어 있어 stable 정렬로 동점을 처리합니다.
    fused_top[start:end] = np.argsort(-scores, axis=1, kind="stable")[:, :TOP_K]

    for user in range(start, end):
        left, right = X.indptr[user:user + 2]
        seen = X.indices[left:right]
        # 이미 본 상품이 전부 앞에 있어도 Top-20을 확보할 수 있는 길이입니다.
        candidates = pop_order[:TOP_K + len(seen)]
        pop_top[user] = candidates[~np.isin(candidates, seen)][:TOP_K]

def make_rankings(top_items):
    return pd.DataFrame({
        USER_COL: np.repeat(eval_user_ids.to_numpy(), TOP_K),
        ITEM_COL: item_ids.to_numpy()[top_items.ravel()],
        "rank": np.tile(np.arange(1, TOP_K + 1), len(eval_user_ids)),
    })

def pooled_recall_at_20(rankings):
    pairs = pd.MultiIndex.from_frame(rankings[[USER_COL, ITEM_COL]])
    hits = int(pairs.isin(truth_pairs).sum())
    return hits / len(truth_pairs), hits

hyerin_val_top20 = make_rankings(fused_top)
pop_recall, pop_hits = pooled_recall_at_20(make_rankings(pop_top))
bm25_recall, bm25_hits = pooled_recall_at_20(hyerin_val_top20)

print(f"Popularity 단독      | Recall@20 {pop_recall:.6f} | 적중 {pop_hits:,}건")
print(f"Popularity + BM25   | Recall@20 {bm25_recall:.6f} | 적중 {bm25_hits:,}건")
print(f"혜린 보고서 대비 차이 | {bm25_recall - 0.042994:+.6f}")
print(f"전체 소요 시간        | {perf_counter() - started:.1f}초")

BM25 이웃 계산 완료 | 2.5초
Popularity 단독      | Recall@20 0.022840 | 적중 3,130건
Popularity + BM25   | Recall@20 0.023737 | 적중 3,253건
혜린 보고서 대비 차이 | -0.019257
전체 소요 시간        | 64.0초


In [5]:
#셀 3. BM25의 IDF 방향과 점수 정규화 비교
from time import perf_counter

started = perf_counter()

# 사용자 한 명의 구매 목록을 문서로 보고 상품별 IDF를 계산합니다.
user_lengths = np.diff(X.indptr).astype(np.float32)
item_freq = np.asarray(X.sum(axis=0)).ravel()

item_idf = np.log1p(
    (X.shape[0] - item_freq + 0.5) / (item_freq + 0.5)
).astype(np.float32)

user_length_norm = K1 * (
    1.0 - B + B * user_lengths / user_lengths.mean()
)

weighted_user_item = X.copy()
weighted_user_item.data = (
    item_idf[weighted_user_item.indices]
    * np.repeat(
        (K1 + 1.0) / (1.0 + user_length_norm),
        np.diff(weighted_user_item.indptr),
    )
).astype(np.float32)

def build_item_neighbors(weighted_items):
    n_items = weighted_items.shape[0]
    weighted_t = weighted_items.T.tocsr()
    norms = np.sqrt(
        np.asarray(weighted_items.multiply(weighted_items).sum(axis=1)).ravel()
    )
    blocks = []

    for start in range(0, n_items, 128):
        stop = min(start + 128, n_items)
        dot_block = (weighted_items[start:stop] @ weighted_t).tocsr()
        all_cols, all_scores, offsets = [], [], [0]

        for row in range(stop - start):
            item = start + row
            left, right = dot_block.indptr[row:row + 2]
            cols = dot_block.indices[left:right]
            dots = dot_block.data[left:right]

            other = cols != item
            cols, dots = cols[other], dots[other]
            scores = dots / ((norms[item] * norms[cols]) ** POWER + SHRINK)

            order = np.lexsort((cols, -scores))[:NEIGHBORS]
            all_cols.append(cols[order])
            all_scores.append(scores[order])
            offsets.append(offsets[-1] + len(order))

        blocks.append(sparse.csr_matrix(
            (np.concatenate(all_scores), np.concatenate(all_cols), offsets),
            shape=(stop - start, n_items),
        ))

    neighbors = sparse.vstack(blocks, format="csr")
    neighbors.sort_indices()
    return neighbors

W_item_idf = build_item_neighbors(weighted_user_item.T.tocsr())
print(f"상품별 IDF 이웃 계산 완료 | {perf_counter() - started:.1f}초", flush=True)

def stable_top20(scores):
    """20위 경계에서 동점이면 상품 열 번호가 작은 순으로 선택합니다."""
    boundary = scores.shape[1] - TOP_K
    cutoffs = np.partition(scores, boundary, axis=1)[:, boundary]
    top = np.empty((scores.shape[0], TOP_K), dtype=np.int32)

    for row, cutoff in enumerate(cutoffs):
        above = np.flatnonzero(scores[row] > cutoff)
        tied = np.flatnonzero(scores[row] == cutoff)
        selected = np.concatenate([above, tied[:TOP_K - len(above)]])
        order = np.lexsort((selected, -scores[row, selected]))
        top[row] = selected[order]
    return top

# log1p는 인기 상위 상품과 나머지 상품 간 점수 격차를 줄입니다.
pop_max = popularity / popularity.max()
pop_log = np.log1p(popularity) / np.log1p(popularity.max())
pop_variants = {"최대값": pop_max, "로그 후 최대값": pop_log}

results = []
best_hits = -1
bm25_debug_val_top20 = None

for idf_label, neighbors in [("사용자별", W), ("상품별", W_item_idf)]:
    top_lists = {
        name: np.empty((X.shape[0], TOP_K), dtype=np.int32)
        for name in ["BM25 단독", *pop_variants]
    }

    for start in range(0, X.shape[0], 128):
        stop = min(start + 128, X.shape[0])
        user_block = X[start:stop]
        scores = (user_block @ neighbors).toarray()
        seen_rows, seen_cols = user_block.nonzero()

        scores[seen_rows, seen_cols] = 0.0
        maxima = scores.max(axis=1, keepdims=True)
        np.divide(scores, maxima, out=scores, where=maxima > 0)

        # BM25 점수가 0인 상품끼리는 상품 ID 순으로 결정합니다.
        scores[seen_rows, seen_cols] = -np.inf
        top_lists["BM25 단독"][start:stop] = stable_top20(scores)

        for pop_label, pop_values in pop_variants.items():
            fused = BM25_WEIGHT * scores + POP_WEIGHT * pop_values
            top_lists[pop_label][start:stop] = stable_top20(fused)

    for pop_label, top_items in top_lists.items():
        rankings = make_rankings(top_items)
        recall, hits = pooled_recall_at_20(rankings)
        results.append({
            "IDF": idf_label,
            "Popularity 정규화": pop_label,
            "Recall@20": recall,
            "적중 수": hits,
            "혜린 점수 대비": recall - 0.042994,
        })

        if hits > best_hits:
            best_hits = hits
            bm25_debug_val_top20 = rankings

    print(f"{idf_label} IDF 평가 완료 | {perf_counter() - started:.1f}초", flush=True)

bm25_debug_results = (
    pd.DataFrame(results)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)
print(bm25_debug_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

상품별 IDF 이웃 계산 완료 | 1.7초
사용자별 IDF 평가 완료 | 25.7초
상품별 IDF 평가 완료 | 50.0초
 IDF Popularity 정규화  Recall@20  적중 수  혜린 점수 대비
 상품별            최대값   0.042841  5871 -0.000153
 상품별        BM25 단독   0.042695  5851 -0.000299
 상품별       로그 후 최대값   0.042651  5845 -0.000343
사용자별            최대값   0.023737  3253 -0.019257
사용자별       로그 후 최대값   0.022854  3132 -0.020140
사용자별        BM25 단독   0.017491  2397 -0.025503


In [6]:
#셀 4. 상품별 IDF 기반 Popularity 가중치 탐색
from time import perf_counter

started = perf_counter()
pop_weights = (0.35, 0.15, 0.25, 0.30, 0.40, 0.45, 0.50)
top_by_weight = {
    weight: np.empty((X.shape[0], TOP_K), dtype=np.int32)
    for weight in pop_weights
}

for start in range(0, X.shape[0], 128):
    stop = min(start + 128, X.shape[0])
    user_block = X[start:stop]
    bm25_scores = (user_block @ W_item_idf).toarray()
    seen_rows, seen_cols = user_block.nonzero()

    bm25_scores[seen_rows, seen_cols] = 0.0
    maxima = bm25_scores.max(axis=1, keepdims=True)
    np.divide(bm25_scores, maxima, out=bm25_scores, where=maxima > 0)
    bm25_scores[seen_rows, seen_cols] = -np.inf

    for pop_weight in pop_weights:
        fused = (1.0 - pop_weight) * bm25_scores + pop_weight * pop_max
        top_by_weight[pop_weight][start:stop] = stable_top20(fused)

rows = []
for pop_weight, top_items in top_by_weight.items():
    recall, hits = pooled_recall_at_20(make_rankings(top_items))
    rows.append({
        "Popularity 가중치": pop_weight,
        "BM25 가중치": 1.0 - pop_weight,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 3 최고 대비 적중": hits - 5_871,
        "혜린 보고서 대비 적중": hits - 5_892,
    })

weight_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)

control_hits = next(
    row["적중 수"] for row in rows if row["Popularity 가중치"] == 0.35
)
assert control_hits == 5_871, f"셀 3 기준점이 달라졌어요: {control_hits:,}건"

best_pop_weight = float(weight_results.loc[0, "Popularity 가중치"])
hyerin_val_top20 = make_rankings(top_by_weight[best_pop_weight])

print(weight_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 Popularity 가중치: {best_pop_weight:.2f}")
print(f"전체 소요 시간: {perf_counter() - started:.1f}초")

 Popularity 가중치  BM25 가중치  Recall@20  적중 수  셀 3 최고 대비 적중  혜린 보고서 대비 적중
       0.400000  0.600000   0.042914  5881            10           -11
       0.300000  0.700000   0.042870  5875             4           -17
       0.350000  0.650000   0.042841  5871             0           -21
       0.250000  0.750000   0.042797  5865            -6           -27
       0.150000  0.850000   0.042783  5863            -8           -29
       0.500000  0.500000   0.042746  5858           -13           -34
       0.450000  0.550000   0.042717  5854           -17           -38

선택한 Popularity 가중치: 0.40
전체 소요 시간: 30.3초


In [7]:
#셀 5. LightGCN 학습과 BM25 앙상블 비교
from time import perf_counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import sparse

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle에서 GPU Accelerator를 켠 뒤 실행하세요.")

started = perf_counter()
device = torch.device("cuda")
rng = np.random.default_rng(42)
torch.manual_seed(42)

n_users, n_items = X.shape
n_nodes = n_users + n_items
DIM = 128
LAYERS = 3
EPOCHS = 575
LR = 0.003
L2 = 1e-4
SAMPLES_PER_EPOCH = 65_536

# 사용자-상품 그래프의 대칭 정규화 가중치 D^(-1/2) A D^(-1/2)
edges = X.tocoo()
positive_users = edges.row
positive_items = edges.col
graph_rows = np.concatenate([positive_users, n_users + positive_items])
graph_cols = np.concatenate([n_users + positive_items, positive_users])
degree = np.bincount(graph_rows, minlength=n_nodes)
edge_weights = (
    degree[graph_rows] * degree[graph_cols]
).astype(np.float32) ** -0.5

graph = sparse.csr_matrix(
    (edge_weights, (graph_rows, graph_cols)), shape=(n_nodes, n_nodes)
)
adjacency = torch.sparse_csr_tensor(
    torch.as_tensor(graph.indptr, dtype=torch.int64, device=device),
    torch.as_tensor(graph.indices, dtype=torch.int64, device=device),
    torch.as_tensor(graph.data, device=device),
    size=graph.shape,
    device=device,
)

embeddings = torch.nn.Embedding(n_nodes, DIM, device=device)
torch.nn.init.normal_(embeddings.weight, std=0.1)
optimizer = torch.optim.Adam(embeddings.parameters(), lr=LR)

def propagate_lightgcn():
    current = embeddings.weight
    total = current
    for _ in range(LAYERS):
        current = torch.sparse.mm(adjacency, current)
        total = total + current
    return total / (LAYERS + 1)

def sample_triplets():
    selected = rng.integers(0, len(positive_users), size=SAMPLES_PER_EPOCH)
    user = positive_users[selected]
    pos = positive_items[selected]
    neg = rng.integers(0, n_items, size=len(selected))

    # 학습에서 본 상품은 음성 샘플에서 제외합니다.
    seen = np.asarray(X[user, neg]).ravel() > 0
    while seen.any():
        neg[seen] = rng.integers(0, n_items, size=seen.sum())
        seen = np.asarray(X[user, neg]).ravel() > 0

    return [
        torch.as_tensor(values, dtype=torch.long, device=device)
        for values in (user, n_users + pos, n_users + neg)
    ]

def lightgcn_top20():
    output = np.empty((n_users, TOP_K), dtype=np.int32)

    with torch.inference_mode():
        vectors = propagate_lightgcn()
        item_vectors = vectors[n_users:]

        for start in range(0, n_users, 256):
            stop = min(start + 256, n_users)
            scores = (vectors[start:stop] @ item_vectors.T).cpu().numpy()
            seen_rows, seen_cols = X[start:stop].nonzero()
            scores[seen_rows, seen_cols] = -np.inf
            output[start:stop] = stable_top20(scores)

    return make_rankings(output)

for epoch in range(1, EPOCHS + 1):
    user, pos, neg = sample_triplets()
    optimizer.zero_grad(set_to_none=True)
    vectors = propagate_lightgcn()

    margin = (vectors[user] * (vectors[pos] - vectors[neg])).sum(dim=1)
    regularization = (
        embeddings(user).square().sum()
        + embeddings(pos).square().sum()
        + embeddings(neg).square().sum()
    ) / (2 * len(user))

    loss = F.softplus(-margin).mean() + L2 * regularization
    loss.backward()
    optimizer.step()

    if epoch == 1 or epoch % 100 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:3d}/{EPOCHS} | loss {loss.item():.5f} | "
            f"{perf_counter() - started:.1f}초",
            flush=True,
        )

lightgcn_val_top20 = lightgcn_top20()

def fuse_rankings(left, right, left_weight, k):
    columns = [USER_COL, ITEM_COL]
    parts = []
    for rankings, weight in ((left, left_weight), (right, 1 - left_weight)):
        part = rankings[columns].copy()
        part["score"] = weight / (k + rankings["rank"].to_numpy())
        parts.append(part)

    scored = (
        pd.concat(parts, ignore_index=True)
        .groupby(columns, as_index=False, sort=False)["score"]
        .sum()
        .sort_values(
            [USER_COL, "score", ITEM_COL],
            ascending=[True, False, True],
        )
    )
    top = scored.groupby(USER_COL, sort=False).head(TOP_K).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    return top[[*columns, "rank"]]

bm25_recall, bm25_hits = pooled_recall_at_20(hyerin_val_top20)
gcn_recall, gcn_hits = pooled_recall_at_20(lightgcn_val_top20)

rows = [
    {"설정": "BM25 + Popularity 기준선", "Recall@20": bm25_recall, "적중 수": bm25_hits},
    {"설정": "LightGCN 단독", "Recall@20": gcn_recall, "적중 수": gcn_hits},
]

best_fusion_hits = -1
for k in (20, 60):
    for bm25_weight in (0.70, 0.60, 0.50):
        fused = fuse_rankings(
            hyerin_val_top20, lightgcn_val_top20, bm25_weight, k
        )
        recall, hits = pooled_recall_at_20(fused)
        rows.append({
            "설정": f"BM25 + LightGCN | BM25 비중 {bm25_weight:.2f} | k={k}",
            "Recall@20": recall,
            "적중 수": hits,
        })
        if hits > best_fusion_hits:
            best_fusion_hits = hits
            bm25_lightgcn_val_top20 = fused

lightgcn_results = pd.DataFrame(rows)
lightgcn_results["BM25 기준선 대비 적중"] = lightgcn_results["적중 수"] - bm25_hits
lightgcn_results = lightgcn_results.sort_values("적중 수", ascending=False)

print(lightgcn_results.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print(f"\n채원 LightGCN 보고서 대비 차이: {gcn_recall - 0.036055:+.6f}")

/tmp/ipykernel_22/1117365641.py:41: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adjacency = torch.sparse_csr_tensor(
/tmp/ipykernel_22/1117365641.py:41: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  adjacency = torch.sparse_csr_tensor(


Epoch   1/575 | loss 0.68843 | 6.3초
Epoch 100/575 | loss 0.42006 | 10.7초
Epoch 200/575 | loss 0.29255 | 15.1초
Epoch 300/575 | loss 0.20003 | 19.5초
Epoch 400/575 | loss 0.13233 | 23.9초
Epoch 500/575 | loss 0.09222 | 28.5초
Epoch 575/575 | loss 0.07337 | 31.8초
                                   설정  Recall@20  적중 수  BM25 기준선 대비 적중
BM25 + LightGCN | BM25 비중 0.60 | k=20   0.043432  5952              71
BM25 + LightGCN | BM25 비중 0.50 | k=20   0.043148  5913              32
BM25 + LightGCN | BM25 비중 0.50 | k=60   0.043148  5913              32
                BM25 + Popularity 기준선   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.70 | k=60   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.70 | k=20   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.60 | k=60   0.042914  5881               0
                          LightGCN 단독   0.035573  4875           -1006

채원 LightGCN 보고서 대비 차이: -0.000482


In [8]:
#셀 5-1. 학습된 LightGCN으로 검증과 앙상블 재개
TOP_K = 20

if "lightgcn_top20" not in globals() or globals().get("epoch") != 575:
    raise RuntimeError(
        "현재 세션에서 575 epoch까지 학습한 LightGCN을 찾을 수 없어요. "
        "아래의 셀 5 수정 방법을 적용한 뒤 학습을 실행하세요."
    )

required = (
    "X", "stable_top20", "make_rankings", "pooled_recall_at_20",
    "hyerin_val_top20", "USER_COL", "ITEM_COL",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"이전 셀에서 준비할 항목이 없어요: {', '.join(missing)}. "
        "현재 세션에서 셀 1~4를 실행한 뒤 이 셀만 다시 실행하세요."
    )

bm25_recall, bm25_hits = pooled_recall_at_20(hyerin_val_top20)
if bm25_hits != 5_881:
    raise ValueError(f"셀 4 BM25 기준선이 달라졌어요: {bm25_hits:,}건")

lightgcn_val_top20 = lightgcn_top20()
gcn_recall, gcn_hits = pooled_recall_at_20(lightgcn_val_top20)

keys = [USER_COL, ITEM_COL]
rows = [
    {"설정": "BM25 + Popularity", "Recall@20": bm25_recall, "적중 수": bm25_hits},
    {"설정": "LightGCN 단독", "Recall@20": gcn_recall, "적중 수": gcn_hits},
]

best_fusion_hits = -1

for k in (20, 60):
    for bm25_weight in (0.70, 0.60, 0.50):
        bm25 = hyerin_val_top20[keys].copy()
        gcn = lightgcn_val_top20[keys].copy()
        bm25["score"] = bm25_weight / (k + hyerin_val_top20["rank"])
        gcn["score"] = (1 - bm25_weight) / (k + lightgcn_val_top20["rank"])

        scores = (
            pd.concat([bm25, gcn], ignore_index=True)
            .groupby(keys, as_index=False, sort=False)["score"]
            .sum()
            .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
        )
        fused = scores.groupby(USER_COL, sort=False).head(TOP_K).copy()
        fused["rank"] = fused.groupby(USER_COL, sort=False).cumcount() + 1
        fused = fused[[*keys, "rank"]]

        recall, hits = pooled_recall_at_20(fused)
        rows.append({
            "설정": f"BM25 + LightGCN | BM25 비중 {bm25_weight:.2f} | k={k}",
            "Recall@20": recall,
            "적중 수": hits,
        })
        if hits > best_fusion_hits:
            best_fusion_hits = hits
            bm25_lightgcn_val_top20 = fused

lightgcn_results = pd.DataFrame(rows).sort_values("적중 수", ascending=False)
lightgcn_results["BM25 기준선 대비 적중"] = lightgcn_results["적중 수"] - bm25_hits

print(lightgcn_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n채원 LightGCN 보고서 대비 차이: {gcn_recall - 0.036055:+.6f}")

                                   설정  Recall@20  적중 수  BM25 기준선 대비 적중
BM25 + LightGCN | BM25 비중 0.60 | k=20   0.043432  5952              71
BM25 + LightGCN | BM25 비중 0.50 | k=20   0.043148  5913              32
BM25 + LightGCN | BM25 비중 0.50 | k=60   0.043148  5913              32
                    BM25 + Popularity   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.70 | k=60   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.70 | k=20   0.042914  5881               0
BM25 + LightGCN | BM25 비중 0.60 | k=60   0.042914  5881               0
                          LightGCN 단독   0.035573  4875           -1006

채원 LightGCN 보고서 대비 차이: -0.000482


In [9]:
#셀 5-2. BM25·LightGCN 검증 결과 저장
from pathlib import Path

output_dir = Path("/kaggle/working")
outputs = {
    "bm25_val_top20.csv": hyerin_val_top20,
    "lightgcn_val_top20.csv": lightgcn_val_top20,
    "bm25_lightgcn_val_top20.csv": bm25_lightgcn_val_top20,
    "lightgcn_results.csv": lightgcn_results,
}

for filename, frame in outputs.items():
    path = output_dir / filename
    frame.to_csv(path, index=False)
    print(f"{filename} | {len(frame):,}행 | {path.stat().st_size:,} bytes")

bm25_val_top20.csv | 720,860행 | 9,887,561 bytes
lightgcn_val_top20.csv | 720,860행 | 9,871,263 bytes
bm25_lightgcn_val_top20.csv | 720,860행 | 9,885,159 bytes
lightgcn_results.csv | 8행 | 562 bytes


In [10]:
#셀 6. SimpleX 학습 및 검증 Top-20 생성
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F

required = (
    "X", "stable_top20", "make_rankings", "pooled_recall_at_20",
    "bm25_lightgcn_val_top20",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"이전 실험에서 필요한 항목을 준비하세요: {', '.join(missing)}")
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU Accelerator를 켜세요.")

TOP_K = 20
SX_DIM = 128
SX_LR = 0.002
SX_EPOCHS = 650
SX_NEGATIVES = 20
SX_GAMMA = 0.5
SX_MARGIN = 0.5
SX_BATCH_SIZE = 8_192

started = perf_counter()
sx_device = torch.device("cuda")
sx_rng = np.random.default_rng(42)
torch.manual_seed(42)

sx_n_users, sx_n_items = X.shape
sx_edges = X.tocoo()
sx_user_degree = np.diff(X.indptr)
sx_history_weights = (
    1.0 / np.maximum(sx_user_degree[sx_edges.row], 1)
).astype(np.float32)

sx_history = torch.sparse_coo_tensor(
    torch.as_tensor(
        np.stack([sx_edges.row, sx_edges.col]),
        dtype=torch.long,
        device=sx_device,
    ),
    torch.as_tensor(sx_history_weights, device=sx_device),
    size=X.shape,
    check_invariants=True,
).coalesce()

sx_users = torch.nn.Embedding(sx_n_users, SX_DIM, device=sx_device)
sx_items = torch.nn.Embedding(sx_n_items, SX_DIM, device=sx_device)
torch.nn.init.normal_(sx_users.weight, std=0.01)
torch.nn.init.normal_(sx_items.weight, std=0.01)

sx_optimizer = torch.optim.Adam(
    [*sx_users.parameters(), *sx_items.parameters()], lr=SX_LR
)

def simplex_vectors():
    history_mean = torch.sparse.mm(sx_history, sx_items.weight)
    user_vectors = SX_GAMMA * sx_users.weight + (1 - SX_GAMMA) * history_mean
    return F.normalize(user_vectors, dim=1), F.normalize(sx_items.weight, dim=1)

def simplex_batch():
    chosen = sx_rng.integers(0, sx_edges.nnz, size=SX_BATCH_SIZE)
    user = sx_edges.row[chosen]
    pos = sx_edges.col[chosen]
    neg_users = np.repeat(user, SX_NEGATIVES)
    neg = sx_rng.integers(0, sx_n_items, size=len(neg_users))

    seen = np.asarray(X[neg_users, neg]).ravel() > 0
    while seen.any():
        neg[seen] = sx_rng.integers(0, sx_n_items, size=int(seen.sum()))
        seen = np.asarray(X[neg_users, neg]).ravel() > 0

    return tuple(
        torch.as_tensor(values, dtype=torch.long, device=sx_device)
        for values in (user, pos, neg.reshape(-1, SX_NEGATIVES))
    )

for sx_epoch in range(1, SX_EPOCHS + 1):
    user, pos, neg = simplex_batch()
    sx_optimizer.zero_grad(set_to_none=True)
    all_user_vectors, all_item_vectors = simplex_vectors()

    user_vectors = all_user_vectors[user]
    pos_cos = (user_vectors * all_item_vectors[pos]).sum(dim=1)
    neg_cos = torch.bmm(
        all_item_vectors[neg], user_vectors.unsqueeze(2)
    ).squeeze(2)

    loss = (1 - pos_cos).mean() + F.relu(neg_cos - SX_MARGIN).mean()
    loss.backward()
    sx_optimizer.step()

    if sx_epoch == 1 or sx_epoch % 100 == 0 or sx_epoch == SX_EPOCHS:
        print(
            f"Epoch {sx_epoch:3d}/{SX_EPOCHS} | loss {loss.item():.5f} | "
            f"{perf_counter() - started:.1f}초",
            flush=True,
        )

sx_top_items = np.empty((sx_n_users, TOP_K), dtype=np.int32)
with torch.inference_mode():
    user_vectors, item_vectors = simplex_vectors()
    for start in range(0, sx_n_users, 256):
        stop = min(start + 256, sx_n_users)
        scores = (user_vectors[start:stop] @ item_vectors.T).cpu().numpy()
        seen_rows, seen_cols = X[start:stop].nonzero()
        scores[seen_rows, seen_cols] = -np.inf
        sx_top_items[start:stop] = stable_top20(scores)

simplex_val_top20 = make_rankings(sx_top_items)
sx_recall, sx_hits = pooled_recall_at_20(simplex_val_top20)
print(f"SimpleX 단독 | Recall@20 {sx_recall:.6f} | 적중 {sx_hits:,}건")
print(f"채원 SimpleX 보고서 대비 차이: {sx_recall - 0.037113:+.6f}")

Epoch   1/650 | loss 0.93190 | 0.3초
Epoch 100/650 | loss 0.30161 | 3.4초
Epoch 200/650 | loss 0.27837 | 6.5초
Epoch 300/650 | loss 0.26991 | 9.5초
Epoch 400/650 | loss 0.26679 | 12.6초
Epoch 500/650 | loss 0.26282 | 15.6초
Epoch 600/650 | loss 0.26256 | 18.7초
Epoch 650/650 | loss 0.26117 | 20.2초
SimpleX 단독 | Recall@20 0.019717 | 적중 2,702건
채원 SimpleX 보고서 대비 차이: -0.017396


In [11]:
#셀 7. BM25·LightGCN 기준선에 SimpleX 추가
from pathlib import Path

import pandas as pd

TOP_K = 20
RRF_K = 20
keys = [USER_COL, ITEM_COL]

base_recall, base_hits = pooled_recall_at_20(bm25_lightgcn_val_top20)
if base_hits != 5_952:
    raise ValueError(f"BM25 + LightGCN 기준선을 확인하세요. 현재 {base_hits:,}건")

rows = [{
    "설정": "BM25 + LightGCN 기준선",
    "SimpleX 비중": 0.0,
    "Recall@20": base_recall,
    "적중 수": base_hits,
}]
next_ensemble_val_top20 = bm25_lightgcn_val_top20
best_hits = base_hits

for sx_weight in (0.05, 0.10, 0.20, 0.30):
    parts = []
    for rankings, weight in (
        (bm25_lightgcn_val_top20, 1 - sx_weight),
        (simplex_val_top20, sx_weight),
    ):
        part = rankings[keys].copy()
        part["score"] = weight / (RRF_K + rankings["rank"])
        parts.append(part)

    scores = (
        pd.concat(parts, ignore_index=True)
        .groupby(keys, as_index=False, sort=False)["score"]
        .sum()
        .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
    )
    fused = scores.groupby(USER_COL, sort=False).head(TOP_K).copy()
    fused["rank"] = fused.groupby(USER_COL, sort=False).cumcount() + 1
    fused = fused[[*keys, "rank"]]

    recall, hits = pooled_recall_at_20(fused)
    rows.append({
        "설정": "BM25 + LightGCN + SimpleX",
        "SimpleX 비중": sx_weight,
        "Recall@20": recall,
        "적중 수": hits,
    })

    if hits > best_hits:
        best_hits = hits
        next_ensemble_val_top20 = fused

simplex_ensemble_results = (
    pd.DataFrame(rows).sort_values("적중 수", ascending=False, kind="stable")
)
simplex_ensemble_results["기준선 대비 적중"] = simplex_ensemble_results["적중 수"] - base_hits
print(simplex_ensemble_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

output_dir = Path("/kaggle/working")
simplex_val_top20.to_csv(output_dir / "simplex_val_top20.csv", index=False)
next_ensemble_val_top20.to_csv(output_dir / "best_ensemble_after_simplex.csv", index=False)
simplex_ensemble_results.to_csv(output_dir / "simplex_ensemble_results.csv", index=False)

                       설정  SimpleX 비중  Recall@20  적중 수  기준선 대비 적중
      BM25 + LightGCN 기준선    0.000000   0.043432  5952          0
BM25 + LightGCN + SimpleX    0.050000   0.043432  5952          0
BM25 + LightGCN + SimpleX    0.100000   0.043432  5952          0
BM25 + LightGCN + SimpleX    0.200000   0.043432  5952          0
BM25 + LightGCN + SimpleX    0.300000   0.043432  5952          0


In [12]:
#셀 8. BM25·LightGCN·SimpleX 가중 RRF 직접 결합
import numpy as np
import pandas as pd

required = (
    "hyerin_val_top20", "lightgcn_val_top20", "simplex_val_top20",
    "bm25_lightgcn_val_top20", "pooled_recall_at_20", "USER_COL", "ITEM_COL",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"이전 셀에서 필요한 결과를 준비하세요: {', '.join(missing)}")

TOP_K = 20
RRF_K = 20
keys = [USER_COL, ITEM_COL]

base = bm25_lightgcn_val_top20
base_recall, base_hits = pooled_recall_at_20(base)
base_pairs = pd.MultiIndex.from_frame(base[keys])
simplex_pairs = pd.MultiIndex.from_frame(simplex_val_top20[keys])

rows = []
best_hits = base_hits
best_after_direct_fusion = base

for sx_weight in (0.0, 0.10, 0.20, 0.30, 0.40):
    # SimpleX 비중을 뺀 나머지는 기존 BM25:LightGCN = 60:40으로 배분합니다.
    bm25_weight = 0.60 * (1 - sx_weight)
    gcn_weight = 0.40 * (1 - sx_weight)
    parts = []

    for rankings, weight in (
        (hyerin_val_top20, bm25_weight),
        (lightgcn_val_top20, gcn_weight),
        (simplex_val_top20, sx_weight),
    ):
        if weight > 0:
            part = rankings[keys].copy()
            part["score"] = weight / (RRF_K + rankings["rank"])
            parts.append(part)

    scores = (
        pd.concat(parts, ignore_index=True)
        .groupby(keys, sort=False, as_index=False)["score"]
        .sum()
        .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
    )
    top = scores.groupby(USER_COL, sort=False).head(TOP_K).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    top = top[[*keys, "rank"]]

    pairs = pd.MultiIndex.from_frame(top[keys])
    added = ~pairs.isin(base_pairs)
    recall, hits = pooled_recall_at_20(top)

    if sx_weight == 0.0 and (added.any() or hits != base_hits):
        raise ValueError(
            f"0% 대조군이 기준선을 재현하지 못했어요. "
            f"기준선 {base_hits:,}건 / 대조군 {hits:,}건. 셀 5-1 결과를 확인하세요."
        )

    rows.append({
        "BM25 비중": bm25_weight,
        "LightGCN 비중": gcn_weight,
        "SimpleX 비중": sx_weight,
        "Recall@20": recall,
        "적중 수": hits,
        "기준선 대비 적중": hits - base_hits,
        "Top-20 신규 편입": int(np.count_nonzero(added)),
        "신규 편입 중 SimpleX 목록에 있던 건수": int(
            np.count_nonzero(added & pairs.isin(simplex_pairs))
        ),
    })

    if hits > best_hits:
        best_hits = hits
        best_after_direct_fusion = top

direct_fusion_results = pd.DataFrame(rows)
print(direct_fusion_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n최고 적중: {best_hits:,}건 | 기준선 대비 {best_hits - base_hits:+,}건")

 BM25 비중  LightGCN 비중  SimpleX 비중  Recall@20  적중 수  기준선 대비 적중  Top-20 신규 편입  신규 편입 중 SimpleX 목록에 있던 건수
0.600000     0.400000    0.000000   0.043432  5952          0             0                          0
0.540000     0.360000    0.100000   0.043841  6008         56         28271                      28271
0.480000     0.320000    0.200000   0.044074  6040         88         56658                      56596
0.420000     0.280000    0.300000   0.043038  5898        -54        165158                     165158
0.360000     0.240000    0.400000   0.039426  5403       -549        352434                     352434

최고 적중: 6,040건 | 기준선 대비 +88건


In [13]:
#셀 9. Implicit ALS 학습 및 검증 Top-20 생성
import os
import subprocess
import sys
from time import perf_counter

import numpy as np
from threadpoolctl import threadpool_limits

TOP_K = 20
required = ("X", "stable_top20", "make_rankings", "pooled_recall_at_20")
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"셀 1~3에서 필요한 항목을 준비하세요: {', '.join(missing)}")

try:
    import implicit
except ModuleNotFoundError as exc:
    if exc.name != "implicit":
        raise
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "implicit==0.7.3"]
    )
    import implicit

from implicit.als import AlternatingLeastSquares

started = perf_counter()
als_model = AlternatingLeastSquares(
    factors=32,
    regularization=0.01,
    alpha=40.0,
    iterations=20,
    random_state=42,
    use_gpu=False,
    num_threads=min(4, os.cpu_count() or 1),
)

als_top_items = np.empty((X.shape[0], TOP_K), dtype=np.int32)

with threadpool_limits(limits=1, user_api="blas"):
    # X는 이진 행렬이에요. alpha는 모델 안에서 한 번 적용합니다.
    als_model.fit(X, show_progress=True)

    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        scores = als_model.user_factors[start:stop] @ als_model.item_factors.T
        seen_rows, seen_cols = X[start:stop].nonzero()
        scores[seen_rows, seen_cols] = -np.inf
        als_top_items[start:stop] = stable_top20(scores)

als_val_top20 = make_rankings(als_top_items)
als_recall, als_hits = pooled_recall_at_20(als_val_top20)

print(f"implicit 버전: {implicit.__version__}")
print(f"ALS 단독 | Recall@20 {als_recall:.6f} | 적중 {als_hits:,}건")
print(f"이전 ALS 실험 대비 적중: {als_hits - 4_745:+,}건")
print(f"소요 시간: {perf_counter() - started:.1f}초")

/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of '/usr/local/lib/python3.13/dist-packages/implicit/gpu/_cuda.so: undefined symbol: _ZN3rmm13device_bufferC1EmNS_16cuda_stream_viewENS_6detail23cccl_async_resource_refIN4cuda2mr3__412resource_refIJNS6_17device_accessibleEEEEEE'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

implicit 버전: 0.7.3
ALS 단독 | Recall@20 0.034625 | 적중 4,745건
이전 ALS 실험 대비 적중: +0건
소요 시간: 9.5초


In [14]:
#셀 10. BM25·LightGCN·SimpleX·ALS 가중 RRF 비교
from pathlib import Path

import pandas as pd

required = (
    "hyerin_val_top20",
    "lightgcn_val_top20",
    "simplex_val_top20",
    "als_val_top20",
    "best_after_direct_fusion",
    "pooled_recall_at_20",
    "USER_COL",
    "ITEM_COL",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"이전 셀의 실행 결과가 필요해요: {', '.join(missing)}"
    )

TOP_K = 20
RRF_K = 20
keys = [USER_COL, ITEM_COL]

base = best_after_direct_fusion
_, base_hits = pooled_recall_at_20(base)
if base_hits != 6_038:
    raise ValueError(f"셀 8 최고 기준선을 확인하세요. 현재 {base_hits:,}건")

base_pairs = pd.MultiIndex.from_frame(base[keys])

# 셀 8의 최고 설정과 같은 식으로 계산합니다.
base_simplex_weight = 0.20
base_bm25_weight = 0.60 * (1.0 - base_simplex_weight)
base_gcn_weight = 0.40 * (1.0 - base_simplex_weight)

rows = []
best_hits = base_hits
best_als_weight = 0.0
best_after_als_val_top20 = base

for als_weight in (0.0, 0.05, 0.10, 0.15, 0.20, 0.25):
    remaining = 1.0 - als_weight
    bm25_weight = base_bm25_weight * remaining
    gcn_weight = base_gcn_weight * remaining
    simplex_weight = base_simplex_weight * remaining

    models = (
        (hyerin_val_top20, bm25_weight),
        (lightgcn_val_top20, gcn_weight),
        (simplex_val_top20, simplex_weight),
        (als_val_top20, als_weight),
    )

    parts = []
    for rankings, weight in models:
        if weight > 0:
            part = rankings[keys].copy()
            part["score"] = weight / (RRF_K + rankings["rank"])
            parts.append(part)

    scores = (
        pd.concat(parts, ignore_index=True)
        .groupby(keys, as_index=False, sort=False)["score"]
        .sum()
        .sort_values(
            [USER_COL, "score", ITEM_COL],
            ascending=[True, False, True],
        )
    )

    top = scores.groupby(USER_COL, sort=False).head(TOP_K).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    top = top[[*keys, "rank"]]

    recall, hits = pooled_recall_at_20(top)
    pairs = pd.MultiIndex.from_frame(top[keys])
    added = int((~pairs.isin(base_pairs)).sum())

    if als_weight == 0.0 and (
        len(pairs) != len(base_pairs) or added != 0 or hits != base_hits
    ):
        raise ValueError(
            f"ALS 0% 대조군 불일치 | 추천 행 {len(pairs):,}/{len(base_pairs):,} | "
            f"신규 편입 {added:,}건 | 적중 {hits:,}/{base_hits:,}건"
        )

    rows.append({
        "BM25 비중": bm25_weight,
        "LightGCN 비중": gcn_weight,
        "SimpleX 비중": simplex_weight,
        "ALS 비중": als_weight,
        "Recall@20": recall,
        "적중 수": hits,
        "기준선 대비 적중": hits - base_hits,
        "Top-20 신규 편입": added,
    })

    if hits > best_hits:
        best_hits = hits
        best_als_weight = als_weight
        best_after_als_val_top20 = top

als_ensemble_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print(
    als_ensemble_results.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)
print(f"\n선택한 ALS 비중 {best_als_weight:.0%} | 최고 적중 {best_hits:,}건")

output_dir = Path("/kaggle/working")
als_val_top20.to_csv(output_dir / "als_val_top20.csv", index=False)
best_after_als_val_top20.to_csv(
    output_dir / "best_ensemble_after_als.csv", index=False
)
als_ensemble_results.to_csv(output_dir / "als_ensemble_results.csv", index=False)

ValueError: 셀 8 최고 기준선을 확인하세요. 현재 6,040건

In [ ]:
#셀 11. ALS 가중치 상한 확장
from pathlib import Path

import pandas as pd

keys = [USER_COL, ITEM_COL]
rrf_k = 20
top_k = 20

baseline_recall, baseline_hits = pooled_recall_at_20(best_after_als_val_top20)
if best_als_weight != 0.25 or baseline_hits != 6_187:
    raise ValueError("셀 10에서 선택된 ALS 25% / 6,187건 결과를 확인하세요.")

best_cell11_top20 = best_after_als_val_top20
best_cell11_hits = baseline_hits
best_cell11_als_weight = best_als_weight

sources = (
    (hyerin_val_top20, base_bm25_weight),
    (lightgcn_val_top20, base_gcn_weight),
    (simplex_val_top20, base_simplex_weight),
)

rows = [{
    "BM25 비중": base_bm25_weight * 0.75,
    "LightGCN 비중": base_gcn_weight * 0.75,
    "SimpleX 비중": base_simplex_weight * 0.75,
    "ALS 비중": 0.25,
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "셀 10 최고 대비 적중": 0,
}]

for als_weight in (0.275, 0.30, 0.325, 0.35, 0.40):
    weights = [weight * (1.0 - als_weight) for _, weight in sources]
    rankings_list = [rankings for rankings, _ in sources] + [als_val_top20]

    parts = []
    for rankings, weight in zip(rankings_list, [*weights, als_weight]):
        part = rankings[keys].copy()
        part["score"] = weight / (rrf_k + rankings["rank"])
        parts.append(part)

    scores = (
        pd.concat(parts, ignore_index=True)
        .groupby(keys, as_index=False, sort=False)["score"]
        .sum()
        .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
    )
    top = scores.groupby(USER_COL, sort=False).head(top_k).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    top = top[[*keys, "rank"]]

    recall, hits = pooled_recall_at_20(top)
    rows.append({
        "BM25 비중": weights[0],
        "LightGCN 비중": weights[1],
        "SimpleX 비중": weights[2],
        "ALS 비중": als_weight,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 10 최고 대비 적중": hits - baseline_hits,
    })

    if hits > best_cell11_hits:
        best_cell11_hits = hits
        best_cell11_als_weight = als_weight
        best_cell11_top20 = top

als_extended_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print(als_extended_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 ALS 비중: {best_cell11_als_weight:.1%} | 최고 적중: {best_cell11_hits:,}건")

output_dir = Path("/kaggle/working")
als_extended_results.to_csv(output_dir / "als_extended_results.csv", index=False)
best_cell11_top20.to_csv(output_dir / "best_ensemble_cell11.csv", index=False)

In [ ]:
#셀 12. ALS 25% 고정 후 BM25·LightGCN·SimpleX 비중 조정
from pathlib import Path

import pandas as pd

keys = [USER_COL, ITEM_COL]
model_names = ["BM25", "LightGCN", "SimpleX"]
model_rankings = [hyerin_val_top20, lightgcn_val_top20, simplex_val_top20]

als_weight = 0.25
rrf_k = 20
top_k = 20
step = 0.03

# 셀 10의 최고 설정과 같은 계산식으로 가중치를 가져옵니다.
base_weights = [
    weight * (1.0 - als_weight)
    for weight in (base_bm25_weight, base_gcn_weight, base_simplex_weight)
]

baseline = best_cell11_top20
baseline_recall, baseline_hits = pooled_recall_at_20(baseline)
if best_cell11_als_weight != 0.25 or baseline_hits != 6_187:
    raise ValueError("셀 11에서 ALS 25% / 6,187건이 선택됐는지 확인하세요.")

def result_row(label, weights, recall, hits):
    return {
        "설정": label,
        **{f"{name} 비중": weight for name, weight in zip(model_names, weights)},
        "ALS 비중": als_weight,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 11 최고 대비 적중": hits - baseline_hits,
    }

rows = [result_row("셀 11 기준선", base_weights, baseline_recall, baseline_hits)]
best_cell12_top20 = baseline
best_cell12_hits = baseline_hits
best_cell12_weights = (*base_weights, als_weight)

# 후보마다 다시 나누지 않고 각 모델의 순위별 RRF 기본 점수를 미리 계산합니다.
rankings_list = [*model_rankings, als_val_top20]
rank_scores = [1.0 / (rrf_k + rankings["rank"]) for rankings in rankings_list]

for source, source_name in enumerate(model_names):
    for target, target_name in enumerate(model_names):
        if source == target:
            continue

        weights = base_weights.copy()
        weights[source] -= step
        weights[target] += step

        parts = []
        for rankings, rank_score, weight in zip(
            rankings_list, rank_scores, [*weights, als_weight]
        ):
            part = rankings[keys].copy()
            part["score"] = weight * rank_score
            parts.append(part)

        scores = (
            pd.concat(parts, ignore_index=True)
            .groupby(keys, sort=False, as_index=False)["score"]
            .sum()
            .sort_values(
                [USER_COL, "score", ITEM_COL],
                ascending=[True, False, True],
            )
        )
        top = scores.groupby(USER_COL, sort=False).head(top_k).copy()
        top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
        top = top[[*keys, "rank"]]

        recall, hits = pooled_recall_at_20(top)
        rows.append(result_row(f"{source_name} → {target_name} 3%p", weights, recall, hits))

        if hits > best_cell12_hits:
            best_cell12_hits = hits
            best_cell12_top20 = top
            best_cell12_weights = (*weights, als_weight)

cell12_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print(cell12_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

output_dir = Path("/kaggle/working")
cell12_results.to_csv(output_dir / "ensemble_weight_results_cell12.csv", index=False)
best_cell12_top20.to_csv(output_dir / "best_ensemble_cell12.csv", index=False)

In [ ]:
#셀 13. 최고 앙상블의 RRF k 탐색
from pathlib import Path

import pandas as pd

keys = [USER_COL, ITEM_COL]
rankings_list = [
    hyerin_val_top20,
    lightgcn_val_top20,
    simplex_val_top20,
    als_val_top20,
]

# 출력 표의 반올림된 숫자 대신 셀 12에서 선택한 값을 사용합니다.
weights = best_cell12_weights
baseline = best_cell12_top20
_, baseline_hits = pooled_recall_at_20(baseline)
baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])

rows = []
best_cell13_k = 20
best_cell13_hits = baseline_hits
best_cell13_top20 = baseline

for rrf_k in (20, 5, 10, 15, 30, 40, 60):
    parts = []
    for rankings, weight in zip(rankings_list, weights):
        part = rankings[keys].copy()
        # 셀 12와 같은 연산 순서를 유지합니다.
        rank_score = 1.0 / (rrf_k + rankings["rank"])
        part["score"] = weight * rank_score
        parts.append(part)

    scores = (
        pd.concat(parts, ignore_index=True)
        .groupby(keys, sort=False, as_index=False)["score"]
        .sum()
        .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
    )
    top = scores.groupby(USER_COL, sort=False).head(20).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    top = top[[*keys, "rank"]]

    recall, hits = pooled_recall_at_20(top)
    pairs = pd.MultiIndex.from_frame(top[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    if rrf_k == 20 and (
        len(pairs) != len(baseline_pairs) or added != 0 or hits != baseline_hits
    ):
        raise ValueError(
            f"k=20 대조군 불일치 | 적중 {hits:,}/{baseline_hits:,}건 | "
            f"신규 편입 {added:,}건"
        )

    rows.append({
        "RRF k": rrf_k,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 12 최고 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })

    if hits > best_cell13_hits:
        best_cell13_k = rrf_k
        best_cell13_hits = hits
        best_cell13_top20 = top

cell13_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)

print(cell13_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 RRF k: {best_cell13_k} | 최고 적중: {best_cell13_hits:,}건")

output_dir = Path("/kaggle/working")
cell13_results.to_csv(output_dir / "ensemble_rrf_k_results_cell13.csv", index=False)
best_cell13_top20.to_csv(output_dir / "best_ensemble_cell13.csv", index=False)

In [ ]:
#셀 14. 모델별 후보 확장과 Recall@20 상한 진단
from time import perf_counter

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

required = (
    "X", "W_item_idf", "pop_max", "best_pop_weight", "als_model",
    "eval_user_ids", "item_ids", "truth_pairs", "USER_COL", "ITEM_COL",
    "hyerin_val_top20", "lightgcn_val_top20", "simplex_val_top20",
    "als_val_top20", "best_cell13_top20", "pooled_recall_at_20",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"이전 셀의 실행 결과가 필요해요: {', '.join(missing)}")

started = perf_counter()
keys = [USER_COL, ITEM_COL]
output_k = 20
max_depth = 100
user_values = eval_user_ids.to_numpy()
item_values = item_ids.to_numpy()

_, actual_hits = pooled_recall_at_20(best_cell13_top20)
if actual_hits != 6_198:
    raise ValueError(f"셀 13 기준선을 확인하세요. 현재 적중 {actual_hits:,}건")

# Recall@20 >= 0.05에 필요한 적중 수를 정수 연산으로 계산합니다.
target_hits = (len(truth_pairs) + 19) // 20

def expanded_top_items(model):
    output = np.empty((X.shape[0], max_depth), dtype=np.int32)

    with threadpool_limits(limits=1, user_api="blas"):
        for start in range(0, X.shape[0], 256):
            stop = min(start + 256, X.shape[0])
            block = X[start:stop]
            seen_rows, seen_cols = block.nonzero()

            if model == "BM25":
                scores = (block @ W_item_idf).toarray()
                scores[seen_rows, seen_cols] = 0.0
                maxima = scores.max(axis=1, keepdims=True)
                np.divide(scores, maxima, out=scores, where=maxima > 0)
                scores = (1.0 - best_pop_weight) * scores + best_pop_weight * pop_max
            else:
                scores = als_model.user_factors[start:stop] @ als_model.item_factors.T

            scores[seen_rows, seen_cols] = -np.inf

            # 점수가 같으면 기존 상품 열 순서로 결정합니다.
            cutoff = np.partition(scores, -max_depth, axis=1)[:, -max_depth]
            if not np.isfinite(cutoff).all():
                raise ValueError(f"{model}의 미구매 후보가 100개 미만인 사용자가 있어요.")

            for row, threshold in enumerate(cutoff):
                above = np.flatnonzero(scores[row] > threshold)
                tied = np.flatnonzero(scores[row] == threshold)
                selected = np.concatenate([above, tied[:max_depth - len(above)]])
                order = np.lexsort((selected, -scores[row, selected]))
                output[start + row] = selected[order]

    return output

def model_hit_pairs(rankings):
    pairs = pd.MultiIndex.from_frame(rankings[keys])
    return pairs[pairs.isin(truth_pairs)].unique()

def expanded_hit_pairs(top_items, depth):
    pairs = pd.MultiIndex.from_arrays([
        np.repeat(user_values, depth),
        item_values[top_items[:, :depth].ravel()],
    ])
    return pairs[pairs.isin(truth_pairs)].unique()

bm25_top100 = expanded_top_items("BM25")
print(f"BM25 Top-100 생성 완료 | {perf_counter() - started:.1f}초", flush=True)

als_top100 = expanded_top_items("ALS")
print(f"ALS Top-100 생성 완료 | {perf_counter() - started:.1f}초", flush=True)

# 확장 결과의 첫 20개가 기존 모델별 Top-20과 같은지 확인합니다.
for name, top_items, original in (
    ("BM25", bm25_top100, hyerin_val_top20),
    ("ALS", als_top100, als_val_top20),
):
    first20 = pd.MultiIndex.from_arrays([
        np.repeat(user_values, output_k),
        item_values[top_items[:, :output_k].ravel()],
    ])
    original_pairs = pd.MultiIndex.from_frame(original[keys])
    if len(first20) != len(original_pairs) or not first20.isin(original_pairs).all():
        changed = int((~first20.isin(original_pairs)).sum())
        raise ValueError(f"{name} Top-20 재현 불일치 | 신규 편입 {changed:,}건")

fixed_hits = model_hit_pairs(lightgcn_val_top20).union(
    model_hit_pairs(simplex_val_top20), sort=False
)
bm25_hits_by_depth = {
    20: model_hit_pairs(hyerin_val_top20),
    50: expanded_hit_pairs(bm25_top100, 50),
    100: expanded_hit_pairs(bm25_top100, 100),
}
als_hits_by_depth = {
    20: model_hit_pairs(als_val_top20),
    50: expanded_hit_pairs(als_top100, 50),
    100: expanded_hit_pairs(als_top100, 100),
}

def candidate_upper_bound(bm25_depth, als_depth):
    hits = fixed_hits.union(bm25_hits_by_depth[bm25_depth], sort=False)
    hits = hits.union(als_hits_by_depth[als_depth], sort=False)
    hits_per_user = hits.get_level_values(0).value_counts()
    return len(hits), int(hits_per_user.clip(upper=output_k).sum())

_, baseline_upper = candidate_upper_bound(20, 20)
if baseline_upper < actual_hits:
    raise ValueError("후보 상한이 실제 앙상블 적중보다 낮아요. 입력 목록을 확인하세요.")

rows = []
for bm25_depth, als_depth in ((20, 20), (50, 20), (100, 20),
                              (20, 50), (20, 100), (100, 100)):
    raw_hits, upper_hits = candidate_upper_bound(bm25_depth, als_depth)
    rows.append({
        "BM25 후보 수": bm25_depth,
        "ALS 후보 수": als_depth,
        "후보 내 정답": raw_hits,
        "Top-20 상한 적중": upper_hits,
        "Recall@20 상한": upper_hits / len(truth_pairs),
        "기존 후보 상한 대비": upper_hits - baseline_upper,
        "목표 대비 상한 여유": upper_hits - target_hits,
    })

cell14_results = pd.DataFrame(rows)
print(cell14_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n현재 실제 적중 {actual_hits:,}건 | 목표 적중 {target_hits:,}건")
print(f"전체 소요 시간 {perf_counter() - started:.1f}초")

In [ ]:
#셀 15. BM25·ALS 후보 확장에 따른 실제 Recall@20 비교
from pathlib import Path

import numpy as np
import pandas as pd

keys = [USER_COL, ITEM_COL]
rrf_k = best_cell13_k
weights = best_cell12_weights  # BM25, LightGCN, SimpleX, ALS 순서
baseline = best_cell13_top20
_, baseline_hits = pooled_recall_at_20(baseline)
baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
target_hits = (len(truth_pairs) + 19) // 20

if rrf_k != 20 or baseline_hits != 6_198:
    raise ValueError("셀 13의 기준선 k=20 / 6,198건을 확인하세요.")

def extended_rrf_part(original, top100, depth, weight, tail_scale):
    part = original[keys].copy()
    part["score"] = weight * (1.0 / (rrf_k + original["rank"]))
    if depth == 20:
        return part

    # 기존 Top-20의 순위를 그대로 쓰고, 셀 14에서 계산한 21위 이하를 붙입니다.
    tail_ranks = np.arange(21, depth + 1)
    tail_size = len(tail_ranks)
    tail = pd.DataFrame({
        USER_COL: np.repeat(user_values, tail_size),
        ITEM_COL: item_values[top100[:, 20:depth].ravel()],
        "score": np.tile(
            (weight * (1.0 / (rrf_k + tail_ranks))) * tail_scale,
            len(user_values),
        ),
    })
    return pd.concat([part, tail], ignore_index=True)

def fixed_rrf_part(rankings, weight):
    part = rankings[keys].copy()
    part["score"] = weight * (1.0 / (rrf_k + rankings["rank"]))
    return part

gcn_part = fixed_rrf_part(lightgcn_val_top20, weights[1])
simplex_part = fixed_rrf_part(simplex_val_top20, weights[2])

settings = (
    (20, 20, 1.0),
    (20, 50, 1.0),
    (20, 100, 1.0),
    (50, 20, 1.0),
    (100, 20, 1.0),
    (50, 50, 1.0),
    (100, 100, 1.0),
    (20, 100, 0.5),
    (100, 100, 0.5),
)

rows = []
best_cell15_top20 = baseline
best_cell15_hits = baseline_hits
best_cell15_setting = settings[0]

for bm25_depth, als_depth, tail_scale in settings:
    bm25_part = extended_rrf_part(
        hyerin_val_top20, bm25_top100, bm25_depth, weights[0], tail_scale
    )
    als_part = extended_rrf_part(
        als_val_top20, als_top100, als_depth, weights[3], tail_scale
    )

    scores = (
        pd.concat([bm25_part, gcn_part, simplex_part, als_part], ignore_index=True)
        .groupby(keys, sort=False, as_index=False)["score"]
        .sum()
        .sort_values([USER_COL, "score", ITEM_COL], ascending=[True, False, True])
    )
    top = scores.groupby(USER_COL, sort=False).head(20).copy()
    top["rank"] = top.groupby(USER_COL, sort=False).cumcount() + 1
    top = top[[*keys, "rank"]]

    recall, hits = pooled_recall_at_20(top)
    pairs = pd.MultiIndex.from_frame(top[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    if (bm25_depth, als_depth) == (20, 20) and (
        hits != baseline_hits or added != 0 or len(pairs) != len(baseline_pairs)
    ):
        raise ValueError(f"Top-20 대조군 불일치 | 적중 {hits:,}건 | 신규 편입 {added:,}건")

    rows.append({
        "BM25 후보 수": bm25_depth,
        "ALS 후보 수": als_depth,
        "21위 이하 점수 배율": tail_scale,
        "Recall@20": recall,
        "적중 수": hits,
        "기준선 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })
    print(f"BM25 {bm25_depth} / ALS {als_depth} / 배율 {tail_scale} | {hits:,}건", flush=True)

    if hits > best_cell15_hits:
        best_cell15_hits = hits
        best_cell15_top20 = top
        best_cell15_setting = (bm25_depth, als_depth, tail_scale)

cell15_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)
print(cell15_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n최고 적중 {best_cell15_hits:,}건 | 0.05 목표까지 {max(0, target_hits - best_cell15_hits):,}건")

output_dir = Path("/kaggle/working")
cell15_results.to_csv(output_dir / "candidate_depth_results_cell15.csv", index=False)
best_cell15_top20.to_csv(output_dir / "best_ensemble_cell15.csv", index=False)

In [ ]:
#셀 16. RRF와 모델 원점수를 섞어 후보 재정렬
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits

required = (
    "X", "W_item_idf", "pop_max", "best_pop_weight", "als_model",
    "propagate_lightgcn", "simplex_vectors", "hyerin_val_top20",
    "lightgcn_val_top20", "simplex_val_top20", "als_val_top20",
    "bm25_top100", "als_top100", "best_cell12_weights",
    "best_cell15_top20", "eval_user_ids", "item_ids",
    "make_rankings", "pooled_recall_at_20", "USER_COL", "ITEM_COL",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"이전 셀의 변수 또는 학습 모델이 필요해요: {', '.join(missing)}")

started = perf_counter()
TOP_K = 20
weights = best_cell12_weights
mixes = (0.0, 0.10, 0.25, 0.50, 0.75, 1.0)
keys = [USER_COL, ITEM_COL]

baseline = best_cell15_top20
_, baseline_hits = pooled_recall_at_20(baseline)
if baseline_hits != 6_247:
    raise ValueError(f"셀 15 최고 기준선을 확인하세요. 현재 {baseline_hits:,}건")

def ranking_columns(frame):
    ordered = (
        frame.set_index([USER_COL, "rank"])[ITEM_COL]
        .unstack("rank")
        .reindex(index=eval_user_ids, columns=range(1, TOP_K + 1))
    )
    columns = item_ids.get_indexer(ordered.to_numpy().ravel())
    if (columns < 0).any():
        raise ValueError("추천 목록에서 누락된 사용자·순위 또는 상품 ID가 있어요.")
    return columns.reshape(len(eval_user_ids), TOP_K)

def attach_tail(frame, top100):
    # 셀 15와 동일하게 기존 Top-20 순위 뒤에 21~100위를 붙입니다.
    return np.concatenate([ranking_columns(frame), top100[:, 20:100]], axis=1)

candidate_lists = (
    attach_tail(hyerin_val_top20, bm25_top100),
    ranking_columns(lightgcn_val_top20),
    ranking_columns(simplex_val_top20),
    attach_tail(als_val_top20, als_top100),
)
rrf_terms = [
    weight * (1.0 / (20 + np.arange(1, candidates.shape[1] + 1)))
    for weight, candidates in zip(weights, candidate_lists)
]

def minmax(values):
    low = values.min()
    span = values.max() - low
    if span == 0:
        return np.zeros_like(values)
    return (values - low) / span

top_by_mix = {
    mix: np.empty((X.shape[0], TOP_K), dtype=np.int32) for mix in mixes
}

with torch.inference_mode():
    gcn_vectors = propagate_lightgcn()
    gcn_users = gcn_vectors[:X.shape[0]]
    gcn_items = gcn_vectors[X.shape[0]:]
    sx_users_eval, sx_items_eval = simplex_vectors()

with torch.inference_mode(), threadpool_limits(limits=1, user_api="blas"):
    for start in range(0, X.shape[0], 128):
        stop = min(start + 128, X.shape[0])
        block = X[start:stop]

        bm25_scores = (block @ W_item_idf).toarray()
        seen_rows, seen_cols = block.nonzero()
        bm25_scores[seen_rows, seen_cols] = 0.0
        maxima = bm25_scores.max(axis=1, keepdims=True)
        np.divide(bm25_scores, maxima, out=bm25_scores, where=maxima > 0)
        bm25_scores *= 1.0 - best_pop_weight
        bm25_scores += best_pop_weight * pop_max

        raw_blocks = (
            bm25_scores,
            (gcn_users[start:stop] @ gcn_items.T).cpu().numpy(),
            (sx_users_eval[start:stop] @ sx_items_eval.T).cpu().numpy(),
            als_model.user_factors[start:stop] @ als_model.item_factors.T,
        )

        for row, user in enumerate(range(start, stop)):
            model_candidates = [candidates[user] for candidates in candidate_lists]
            union = np.unique(np.concatenate(model_candidates))

            rrf_scores = np.zeros(len(union), dtype=np.float64)
            raw_scores = np.zeros(len(union), dtype=np.float64)
            for candidates, terms, raw, weight in zip(
                model_candidates, rrf_terms, raw_blocks, weights
            ):
                rrf_scores[np.searchsorted(union, candidates)] += terms
                raw_scores += weight * minmax(raw[row, union].astype(np.float64))

            rrf_norm = minmax(rrf_scores)
            raw_norm = minmax(raw_scores)
            for mix in mixes:
                score = rrf_scores if mix == 0 else (1 - mix) * rrf_norm + mix * raw_norm
                top_by_mix[mix][user] = union[np.lexsort((union, -score))[:TOP_K]]

        if stop == X.shape[0] or stop % 8192 == 0:
            print(f"{stop:,}/{X.shape[0]:,}명 | {perf_counter() - started:.1f}초", flush=True)

baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
rows = []
best_cell16_top20 = baseline
best_cell16_hits = baseline_hits
best_cell16_mix = 0.0

for mix, top_items in top_by_mix.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    rows.append({
        "원점수 비중": mix,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 15 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })

    if mix == 0 and (hits != baseline_hits or added):
        print(
            f"RRF 대조군 점검 | 적중 {hits:,}/{baseline_hits:,}건 | "
            f"신규 편입 {added:,}건. 셀 15와 점수 합산 방식이 달라 "
            "동점 경계의 결과가 달라질 수 있어요. 비교 기준은 셀 15로 유지합니다."
        )

    if mix > 0 and hits > best_cell16_hits:
        best_cell16_hits = hits
        best_cell16_mix = mix
        best_cell16_top20 = rankings

cell16_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell16_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(
    f"\n선택한 원점수 비중 {best_cell16_mix:.0%} | "
    f"최고 적중 {best_cell16_hits:,}건 | "
    f"0.05 목표까지 {max(0, (len(truth_pairs) + 19) // 20 - best_cell16_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell16_results.to_csv(output_dir / "score_rerank_results_cell16.csv", index=False)
best_cell16_top20.to_csv(output_dir / "best_ensemble_cell16.csv", index=False)

In [ ]:
#셀 17. 원점수 혼합 비중 75% 주변 탐색
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits

required = (
    "X", "W_item_idf", "pop_max", "best_pop_weight", "als_model",
    "propagate_lightgcn", "simplex_vectors", "candidate_lists", "rrf_terms",
    "minmax", "best_cell12_weights", "best_cell16_top20", "truth_pairs",
    "make_rankings", "pooled_recall_at_20", "USER_COL", "ITEM_COL",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"셀 16을 실행한 세션에서 다시 실행하세요. 누락 항목: {', '.join(missing)}"
    )

started = perf_counter()
mixes = (0.75, 0.60, 0.65, 0.70, 0.80, 0.85, 0.90)
keys = [USER_COL, ITEM_COL]
weights = best_cell12_weights

baseline = best_cell16_top20
_, baseline_hits = pooled_recall_at_20(baseline)
if baseline_hits != 6_363:
    raise ValueError(f"셀 16의 최고 결과를 확인하세요. 현재 적중: {baseline_hits:,}건")

baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
cell17_top_by_mix = {
    mix: np.empty((X.shape[0], 20), dtype=np.int32)
    for mix in mixes
}

with torch.inference_mode(), threadpool_limits(limits=1, user_api="blas"):
    gcn_vectors = propagate_lightgcn()
    gcn_users = gcn_vectors[:X.shape[0]]
    gcn_items = gcn_vectors[X.shape[0]:]
    simplex_users, simplex_items = simplex_vectors()

    for start in range(0, X.shape[0], 128):
        stop = min(start + 128, X.shape[0])
        block = X[start:stop]

        bm25_scores = (block @ W_item_idf).toarray()
        seen_rows, seen_cols = block.nonzero()
        bm25_scores[seen_rows, seen_cols] = 0.0
        maxima = bm25_scores.max(axis=1, keepdims=True)
        np.divide(bm25_scores, maxima, out=bm25_scores, where=maxima > 0)
        bm25_scores *= 1.0 - best_pop_weight
        bm25_scores += best_pop_weight * pop_max

        raw_blocks = (
            bm25_scores,
            (gcn_users[start:stop] @ gcn_items.T).cpu().numpy(),
            (simplex_users[start:stop] @ simplex_items.T).cpu().numpy(),
            als_model.user_factors[start:stop] @ als_model.item_factors.T,
        )

        for row, user in enumerate(range(start, stop)):
            model_candidates = [candidates[user] for candidates in candidate_lists]
            union = np.unique(np.concatenate(model_candidates))
            rrf_scores = np.zeros(len(union), dtype=np.float64)
            raw_scores = np.zeros(len(union), dtype=np.float64)

            for candidates, terms, raw, weight in zip(
                model_candidates, rrf_terms, raw_blocks, weights
            ):
                rrf_scores[np.searchsorted(union, candidates)] += terms
                raw_scores += weight * minmax(raw[row, union].astype(np.float64))

            rrf_norm = minmax(rrf_scores)
            raw_norm = minmax(raw_scores)

            for mix in mixes:
                score = (1.0 - mix) * rrf_norm + mix * raw_norm
                order = np.lexsort((union, -score))[:20]
                cell17_top_by_mix[mix][user] = union[order]

        if stop % 8192 == 0 or stop == X.shape[0]:
            print(
                f"{stop:,}/{X.shape[0]:,}명 | {perf_counter() - started:.1f}초",
                flush=True,
            )

rows = []
best_cell17_mix = 0.75
best_cell17_hits = baseline_hits
best_cell17_top20 = baseline

for mix, top_items in cell17_top_by_mix.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    if mix == 0.75 and (
        hits != baseline_hits or added != 0 or len(pairs) != len(baseline_pairs)
    ):
        raise ValueError(
            f"셀 16의 75% 기준점 불일치 | "
            f"적중 {hits:,}/{baseline_hits:,}건 | 신규 편입 {added:,}건"
        )

    rows.append({
        "원점수 비중": mix,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 16 최고 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })

    if hits > best_cell17_hits:
        best_cell17_mix = mix
        best_cell17_hits = hits
        best_cell17_top20 = rankings

cell17_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)
print(cell17_results.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 원점수 비중 {best_cell17_mix:.0%} | "
    f"최고 적중 {best_cell17_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell17_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell17_results.to_csv(output_dir / "score_rerank_results_cell17.csv", index=False)
best_cell17_top20.to_csv(output_dir / "best_ensemble_cell17.csv", index=False)

In [ ]:
#셀 18. LightGBM 재정렬용 후보 피처 생성
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits

required = (
    "X", "W_item_idf", "pop_max", "best_pop_weight", "als_model",
    "propagate_lightgcn", "simplex_vectors", "candidate_lists", "rrf_terms",
    "minmax", "best_cell12_weights", "best_cell17_mix", "best_cell17_top20",
    "eval_user_ids", "item_ids", "truth_pairs", "make_rankings",
    "pooled_recall_at_20",
)
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"셀 17까지 실행한 세션이 필요해요. 누락: {', '.join(missing)}")

started = perf_counter()
names = ("bm25", "lightgcn", "simplex", "als")
feature_names = [
    *[f"{name}_raw" for name in names],
    *[f"{name}_normalized" for name in names],
    *[f"{name}_reciprocal_rank" for name in names],
    "model_count", "log_item_popularity", "log_user_history",
    "rrf_normalized", "raw_combined", "cell17_score",
]

n_users = X.shape[0]
max_candidates = sum(candidates.shape[1] for candidates in candidate_lists)
features = np.empty((n_users * max_candidates, len(feature_names)), dtype=np.float32)
candidate_items = np.empty(n_users * max_candidates, dtype=np.int32)
offsets = np.zeros(n_users + 1, dtype=np.int64)

log_popularity = np.log1p(np.asarray(X.sum(axis=0)).ravel())
log_history = np.log1p(np.diff(X.indptr))
weights = best_cell12_weights
mix = best_cell17_mix

with torch.inference_mode(), threadpool_limits(limits=1, user_api="blas"):
    gcn = propagate_lightgcn()
    sx_user_vectors, sx_item_vectors = simplex_vectors()

    for start in range(0, n_users, 128):
        stop = min(start + 128, n_users)
        block = X[start:stop]

        bm25 = (block @ W_item_idf).toarray()
        seen_rows, seen_cols = block.nonzero()
        bm25[seen_rows, seen_cols] = 0.0
        maxima = bm25.max(axis=1, keepdims=True)
        np.divide(bm25, maxima, out=bm25, where=maxima > 0)
        bm25 *= 1.0 - best_pop_weight
        bm25 += best_pop_weight * pop_max

        raw_blocks = (
            bm25,
            (gcn[start:stop] @ gcn[n_users:].T).cpu().numpy(),
            (sx_user_vectors[start:stop] @ sx_item_vectors.T).cpu().numpy(),
            als_model.user_factors[start:stop] @ als_model.item_factors.T,
        )

        for row, user in enumerate(range(start, stop)):
            lists = [candidates[user] for candidates in candidate_lists]
            union = np.unique(np.concatenate(lists))
            begin = offsets[user]
            end = begin + len(union)
            offsets[user + 1] = end
            candidate_items[begin:end] = union

            raw = np.column_stack([
                scores[row, union].astype(np.float64) for scores in raw_blocks
            ])
            normalized = np.column_stack([
                minmax(raw[:, column]) for column in range(len(names))
            ])
            reciprocal = np.zeros_like(raw)
            rrf = np.zeros(len(union), dtype=np.float64)
            combined = np.zeros(len(union), dtype=np.float64)

            for column, (items, terms, weight) in enumerate(
                zip(lists, rrf_terms, weights)
            ):
                positions = np.searchsorted(union, items)
                reciprocal[positions, column] = 1.0 / np.arange(1, len(items) + 1)
                rrf[positions] += terms
                combined += weight * normalized[:, column]

            rrf_norm = minmax(rrf)
            combined_norm = minmax(combined)
            cell17_score = (1.0 - mix) * rrf_norm + mix * combined_norm

            features[begin:end] = np.column_stack([
                raw, normalized, reciprocal,
                (reciprocal > 0).sum(axis=1),
                log_popularity[union],
                np.full(len(union), log_history[user]),
                rrf_norm, combined_norm, cell17_score,
            ])

        if stop % 8192 == 0 or stop == n_users:
            print(f"피처 생성 {stop:,}/{n_users:,}명", flush=True)

features = features[:offsets[-1]]
candidate_items = candidate_items[:offsets[-1]]
group_sizes = np.diff(offsets)
user_values = eval_user_ids.to_numpy()
item_values = item_ids.to_numpy()

# 정답 여부는 학습 라벨로만 사용합니다. 피처에는 포함되지 않아요.
labels = np.empty(len(candidate_items), dtype=np.uint8)
for start in range(0, n_users, 2048):
    stop = min(start + 2048, n_users)
    begin, end = offsets[start], offsets[stop]
    pairs = pd.MultiIndex.from_arrays([
        np.repeat(user_values[start:stop], group_sizes[start:stop]),
        item_values[candidate_items[begin:end]],
    ])
    labels[begin:end] = pairs.isin(truth_pairs)

print(f"후보 {len(labels):,}건 | 후보 내 정답 {labels.sum():,}건")
print(f"피처 {len(feature_names)}개 | 소요 {perf_counter() - started:.1f}초")

In [ ]:
#셀 19. LightGBM 3-fold 재정렬과 셀 17 비교
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
fold_by_user = np.empty(n_users, dtype=np.int8)
fold_by_user[rng.permutation(n_users)] = np.arange(n_users) % 3
fold_by_candidate = np.repeat(fold_by_user, group_sizes)

learned_top = np.empty((n_users, 20), dtype=np.int32)
blended_top = np.empty_like(learned_top)

# 후보에 정답이 하나라도 있는 사용자만 랭킹 학습에 사용합니다.
# 평가할 때는 모든 사용자의 추천을 생성합니다.
has_positive = np.add.reduceat(labels, offsets[:-1]) > 0
positive_user_rows = np.repeat(has_positive, group_sizes)

for fold in range(3):
    train_users = (fold_by_user != fold) & has_positive
    train_rows = (fold_by_candidate != fold) & positive_user_rows
    test_users = np.flatnonzero(fold_by_user == fold)
    test_rows = fold_by_candidate == fold

    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        lambdarank_truncation_level=23,
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=100,
        reg_lambda=1.0,
        n_jobs=4,
        random_state=42,
        verbosity=-1,
    )
    ranker.fit(
        features[train_rows],
        labels[train_rows],
        group=group_sizes[train_users],
        feature_name=feature_names,
    )
    predictions = ranker.predict(features[test_rows])

    cursor = 0
    for user in test_users:
        begin, end = offsets[user:user + 2]
        size = end - begin
        items = candidate_items[begin:end]
        learned = predictions[cursor:cursor + size]
        baseline = features[begin:end, -1]
        blended = 0.50 * minmax(learned) + 0.50 * baseline

        learned_top[user] = items[np.lexsort((items, -learned))[:20]]
        blended_top[user] = items[np.lexsort((items, -blended))[:20]]
        cursor += size

    print(f"Fold {fold + 1}/3 완료 | 평가 사용자 {len(test_users):,}명", flush=True)

baseline_recall, baseline_hits = pooled_recall_at_20(best_cell17_top20)
if baseline_hits != 6_379:
    raise ValueError(f"셀 17 기준선을 확인하세요. 현재 {baseline_hits:,}건")

rows = [{
    "설정": "셀 17 기준선",
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "기준선 대비 적중": 0,
}]

for name, top in (
    ("LightGBM 단독 (OOF)", learned_top),
    ("LightGBM 50% + 셀 17 50% (OOF)", blended_top),
):
    recall, hits = pooled_recall_at_20(make_rankings(top))
    rows.append({
        "설정": name,
        "Recall@20": recall,
        "적중 수": hits,
        "기준선 대비 적중": hits - baseline_hits,
    })

cell19_results = pd.DataFrame(rows).sort_values("적중 수", ascending=False)
print(cell19_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
cell19_results.to_csv(
    Path("/kaggle/working") / "lightgbm_oof_results_cell19.csv", index=False
)

In [ ]:
#셀 20. 내부 분할과 LightGBM 학습용 피처 생성
from IPython import get_ipython

import numpy as np
import pandas as pd

# 이전 셀의 코드를 현재 세션의 실행 기록에서 가져옵니다.
# 내부 실험은 별도 변수 공간에서 실행합니다.
history = tuple(get_ipython().user_ns["In"])

def previous_cell(title):
    for source in reversed(history):
        lines = source.strip().splitlines()
        if lines and lines[0].strip() == title:
            return source
    raise RuntimeError(f"현재 세션의 실행 기록에서 찾지 못했어요: {title}")

source3 = previous_cell("#셀 3. BM25의 IDF 방향과 점수 정규화 비교")
source4 = previous_cell("#셀 4. 상품별 IDF 기반 Popularity 가중치 탐색")
source5 = previous_cell("#셀 5. LightGCN 학습과 BM25 앙상블 비교")
source6 = previous_cell("#셀 6. SimpleX 학습 및 검증 Top-20 생성")
source9 = previous_cell("#셀 9. Implicit ALS 학습 및 검증 Top-20 생성")
source14 = previous_cell("#셀 14. 모델별 후보 확장과 Recall@20 상한 진단")
source16 = previous_cell("#셀 16. RRF와 모델 원점수를 섞어 후보 재정렬")
source18 = previous_cell("#셀 18. LightGBM 재정렬용 후보 피처 생성")

def before_line(source, line_start):
    lines = source.splitlines()
    for index, line in enumerate(lines):
        if line.startswith(line_start):
            return "\n".join(lines[:index])
    raise RuntimeError(f"기존 셀에서 다음 코드 줄을 찾지 못했어요: {line_start}")

def function_source(source, name):
    import ast

    for node in ast.parse(source).body:
        if isinstance(node, ast.FunctionDef) and node.name == name:
            return ast.get_source_segment(source, node)
    raise RuntimeError(f"기존 셀에서 함수를 찾지 못했어요: {name}")

# 학습 전에 재사용할 코드가 모두 있는지 확인합니다.
blocks = {
    "BM25 함수": [
        function_source(source3, "build_item_neighbors"),
        function_source(source3, "stable_top20"),
    ],
    "LightGCN": [before_line(source5, "lightgcn_val_top20 =")],
    "SimpleX": [before_line(source6, "sx_recall, sx_hits =")],
    "ALS": [before_line(source9, "als_recall, als_hits =")],
    "후보 확장 함수": [function_source(source14, "expanded_top_items")],
    "후보 목록 함수": [
        function_source(source16, name)
        for name in ("ranking_columns", "attach_tail", "minmax")
    ],
    "재정렬 피처": [source18],
}
for name, sources in blocks.items():
    for source in sources:
        compile(source, f"<{name}>", "exec")

rng = np.random.default_rng(42)
coo = X.tocoo()
held_out = np.zeros(X.nnz, dtype=bool)

# 사용자마다 최소 1건은 학습에 남깁니다.
for user in range(X.shape[0]):
    begin, end = X.indptr[user:user + 2]
    count = end - begin
    if count >= 2:
        size = min(count - 1, max(1, round(0.20 * count)))
        held_out[rng.choice(np.arange(begin, end), size=size, replace=False)] = True

inner_X = X.copy()
inner_X.data[held_out] = 0
inner_X.eliminate_zeros()

inner_truth_pairs = pd.MultiIndex.from_arrays([
    eval_user_ids.to_numpy()[coo.row[held_out]],
    item_ids.to_numpy()[coo.col[held_out]],
], names=[USER_COL, ITEM_COL])

print(
    f"내부 학습 {inner_X.nnz:,}건 | 내부 정답 {len(inner_truth_pairs):,}건",
    flush=True,
)

# 내부 데이터와 필요한 설정만 넘깁니다.
inner = {
    "X": inner_X,
    "truth_pairs": inner_truth_pairs,
    "eval_user_ids": eval_user_ids,
    "item_ids": item_ids,
    "USER_COL": USER_COL,
    "ITEM_COL": ITEM_COL,
    "TOP_K": 20,
    "best_pop_weight": best_pop_weight,
    "best_cell12_weights": best_cell12_weights,
    "best_cell17_mix": best_cell17_mix,
    "K1": K1,
    "B": B,
    "POWER": POWER,
    "SHRINK": SHRINK,
    "NEIGHBORS": NEIGHBORS,
    "np": np,
    "pd": pd,
    "sparse": sparse,
}

# 기존 함수의 전역 변수 참조도 내부 데이터로 연결합니다.
inner["make_rankings"] = type(make_rankings)(make_rankings.__code__, inner)
inner["pooled_recall_at_20"] = type(pooled_recall_at_20)(
    pooled_recall_at_20.__code__, inner
)

for source in blocks["BM25 함수"]:
    exec(source, inner)

# 셀 3에서 검증한 상품별 IDF 방식으로 내부 BM25 이웃을 계산합니다.
lengths = np.diff(inner_X.indptr).astype(np.float32)
popularity_inner = np.asarray(inner_X.sum(axis=0)).ravel()
idf_inner = np.log1p(
    (inner_X.shape[0] - popularity_inner + 0.5) / (popularity_inner + 0.5)
).astype(np.float32)
length_norm = K1 * (1.0 - B + B * lengths / lengths.mean())

weighted_inner = inner_X.copy()
weighted_inner.data = (
    idf_inner[weighted_inner.indices]
    * np.repeat((K1 + 1.0) / (1.0 + length_norm), np.diff(weighted_inner.indptr))
).astype(np.float32)

inner["W_item_idf"] = inner["build_item_neighbors"](weighted_inner.T.tocsr())
inner["pop_max"] = popularity_inner / popularity_inner.max()

# 기존 학습 셀의 입력 점검을 통과할 수 있도록 내부 BM25 결과를 먼저 만듭니다.
inner.update({
    "max_depth": 100,
    "threadpool_limits": threadpool_limits,
})
exec(blocks["후보 확장 함수"][0], inner)
inner["bm25_top100"] = inner["expanded_top_items"]("BM25")
inner["hyerin_val_top20"] = inner["make_rankings"](inner["bm25_top100"][:, :20])

# SimpleX 학습 셀은 이 변수가 준비됐는지만 확인합니다.
# 실제 내부 앙상블 후보에는 아래에서 생성할 모델별 추천을 사용합니다.
inner["bm25_lightgcn_val_top20"] = inner["hyerin_val_top20"]
inner["best_cell17_top20"] = inner["hyerin_val_top20"]

print("내부 LightGCN 학습", flush=True)
exec(blocks["LightGCN"][0], inner)
inner["lightgcn_val_top20"] = inner["lightgcn_top20"]()

print("내부 SimpleX 학습", flush=True)
exec(blocks["SimpleX"][0], inner)

print("내부 ALS 학습", flush=True)
exec(blocks["ALS"][0], inner)
inner["als_top100"] = inner["expanded_top_items"]("ALS")

for source in blocks["후보 목록 함수"]:
    exec(source, inner)

inner["candidate_lists"] = (
    inner["bm25_top100"],
    inner["ranking_columns"](inner["lightgcn_val_top20"]),
    inner["ranking_columns"](inner["simplex_val_top20"]),
    inner["als_top100"],
)
inner["rrf_terms"] = [
    weight * (1.0 / (20 + np.arange(1, candidates.shape[1] + 1)))
    for weight, candidates in zip(best_cell12_weights, inner["candidate_lists"])
]

print("내부 LightGBM 학습용 피처 생성", flush=True)
exec(blocks["재정렬 피처"][0], inner)

if inner["feature_names"] != feature_names:
    raise ValueError("내부 학습 피처와 공통 Validation 피처의 열 순서가 달라요.")

In [ ]:
#셀 21. 내부 학습 LightGBM의 공통 Validation 평가
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

inner_labels = inner["labels"]
inner_groups = inner["group_sizes"]
inner_has_positive = np.add.reduceat(inner_labels, inner["offsets"][:-1]) > 0
inner_train_rows = np.repeat(inner_has_positive, inner_groups)

ranker_inner = lgb.LGBMRanker(
    objective="lambdarank",
    lambdarank_truncation_level=23,
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=100,
    reg_lambda=1.0,
    n_jobs=4,
    random_state=42,
    verbosity=-1,
)

ranker_inner.fit(
    pd.DataFrame(inner["features"][inner_train_rows], columns=feature_names),
    inner_labels[inner_train_rows],
    group=inner_groups[inner_has_positive],
)

baseline_recall, baseline_hits = pooled_recall_at_20(best_cell17_top20)
if baseline_hits != 6_379:
    raise ValueError(f"셀 17 기준선이 달라졌어요. 현재 {baseline_hits:,}건")

learned_top = np.empty((len(group_sizes), 20), dtype=np.int32)
blended_top = np.empty_like(learned_top)

for start in range(0, len(group_sizes), 2048):
    stop = min(start + 2048, len(group_sizes))
    block_start, block_end = offsets[start], offsets[stop]
    predictions = ranker_inner.predict(
        pd.DataFrame(features[block_start:block_end], columns=feature_names)
    )

    for user in range(start, stop):
        begin, end = offsets[user:user + 2]
        items = candidate_items[begin:end]
        learned = predictions[begin - block_start:end - block_start]
        blended = 0.50 * minmax(learned) + 0.50 * features[begin:end, -1]

        learned_top[user] = items[np.lexsort((items, -learned))[:20]]
        blended_top[user] = items[np.lexsort((items, -blended))[:20]]

rows = [{
    "설정": "셀 17 기준선",
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "기준선 대비 적중": 0,
}]

inner_blended_val_top20 = make_rankings(blended_top)

for name, rankings in (
    ("내부 학습 LightGBM 단독", make_rankings(learned_top)),
    ("내부 학습 LightGBM 50% + 셀 17 50%", inner_blended_val_top20),
):
    recall, hits = pooled_recall_at_20(rankings)
    rows.append({
        "설정": name,
        "Recall@20": recall,
        "적중 수": hits,
        "기준선 대비 적중": hits - baseline_hits,
    })

cell21_results = pd.DataFrame(rows).sort_values("적중 수", ascending=False)
print(cell21_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

output_dir = Path("/kaggle/working")
cell21_results.to_csv(output_dir / "lightgbm_inner_split_results_cell21.csv", index=False)
inner_blended_val_top20.to_csv(output_dir / "lightgbm_inner_blend_val_top20.csv", index=False)

In [ ]:
#셀 22. EASE 학습 및 기존 후보 밖 정답 확인
import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve
from threadpoolctl import threadpool_limits

ease_size = min(3_500, X.shape[1])
ease_l2 = 300.0
ease_k = 20

item_pop = np.asarray(X.sum(axis=0)).ravel()
# 인기 순으로 3,500개를 선택하고, 점수 동점은 기존 상품 ID 순으로 처리합니다.
ease_catalog = np.sort(np.lexsort((np.arange(X.shape[1]), -item_pop))[:ease_size])
ease_X = X[:, ease_catalog].tocsr().astype(np.float64)

ease_top = np.empty((X.shape[0], ease_k), dtype=np.int32)

with threadpool_limits(limits=4, user_api="blas"):
    gram = (ease_X.T @ ease_X).toarray()
    gram.flat[::ease_size + 1] += ease_l2
    factor = cho_factor(gram, overwrite_a=True)
    precision = cho_solve(factor, np.eye(ease_size), overwrite_b=True)

    # EASE의 상품 간 가중치 B[i, j] = -P[i, j] / P[j, j]
    ease_weights = -precision / np.diag(precision)
    np.fill_diagonal(ease_weights, 0.0)
    del gram, factor, precision

    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        block = ease_X[start:stop]
        scores = block @ ease_weights
        seen_rows, seen_cols = block.nonzero()
        scores[seen_rows, seen_cols] = -np.inf

        order = np.argsort(-scores, axis=1, kind="stable")[:, :ease_k]
        if not np.isfinite(np.take_along_axis(scores, order, axis=1)).all():
            raise ValueError("EASE 미구매 후보가 20개 미만인 사용자가 있어요.")
        ease_top[start:stop] = ease_catalog[order]

ease_val_top20 = make_rankings(ease_top)
ease_recall, ease_hits = pooled_recall_at_20(ease_val_top20)

# 셀 18의 후보 중 정답에 해당하는 사용자–상품 쌍만 가져옵니다.
candidate_hit_rows = np.flatnonzero(labels)
candidate_hit_users = np.searchsorted(
    offsets[1:], candidate_hit_rows, side="right"
)
existing_hits = pd.MultiIndex.from_arrays([
    eval_user_ids.to_numpy()[candidate_hit_users],
    item_ids.to_numpy()[candidate_items[candidate_hit_rows]],
])

ease_pairs = pd.MultiIndex.from_frame(ease_val_top20[[USER_COL, ITEM_COL]])
ease_hit_pairs = ease_pairs[ease_pairs.isin(truth_pairs)].unique()
ease_new_hits = int((~ease_hit_pairs.isin(existing_hits)).sum())

print(f"EASE 단독 | Recall@20 {ease_recall:.6f} | 적중 {ease_hits:,}건")
print(f"기존 후보 합집합의 정답: {len(existing_hits):,}건")
print(f"EASE가 추가로 찾은 정답: {ease_new_hits:,}건")
print(f"EASE 포함 후보 합집합의 정답: {len(existing_hits) + ease_new_hits:,}건")

In [ ]:
#셀 23. 기존 후보에 EASE 점수를 섞어 재정렬
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

baseline_recall, baseline_hits = pooled_recall_at_20(best_cell17_top20)
if baseline_hits != 6_379:
    raise ValueError(f"셀 17 기준선을 확인하세요. 현재 {baseline_hits:,}건")

ease_mixes = (0.05, 0.10, 0.15, 0.20)
ease_top_by_mix = {
    mix: np.empty((X.shape[0], 20), dtype=np.int32)
    for mix in ease_mixes
}

# 전체 상품 열 번호를 EASE 카탈로그 안의 열 번호로 매핑합니다.
ease_col = np.full(X.shape[1], -1, dtype=np.int32)
ease_col[ease_catalog] = np.arange(len(ease_catalog), dtype=np.int32)
baseline_score_col = feature_names.index("cell17_score")

with threadpool_limits(limits=4, user_api="blas"):
    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        block_scores = ease_X[start:stop] @ ease_weights

        for user in range(start, stop):
            begin, end = offsets[user:user + 2]
            items = candidate_items[begin:end]
            base_scores = features[begin:end, baseline_score_col]

            local_cols = ease_col[items]
            covered = local_cols >= 0
            ease_scores = np.zeros(len(items), dtype=np.float64)
            if covered.any():
                ease_scores[covered] = minmax(
                    block_scores[user - start, local_cols[covered]]
                )

            for mix in ease_mixes:
                combined = (1.0 - mix) * base_scores + mix * ease_scores
                order = np.lexsort((items, -combined))[:20]
                ease_top_by_mix[mix][user] = items[order]

keys = [USER_COL, ITEM_COL]
base_pairs = pd.MultiIndex.from_frame(best_cell17_top20[keys])

rows = [{
    "EASE 비중": 0.0,
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "셀 17 대비 적중": 0,
    "Top-20 신규 편입": 0,
}]

best_cell23_mix = 0.0
best_cell23_hits = baseline_hits
best_cell23_top20 = best_cell17_top20

for mix, top_items in ease_top_by_mix.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys])

    rows.append({
        "EASE 비중": mix,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 17 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": int((~pairs.isin(base_pairs)).sum()),
    })

    if hits > best_cell23_hits:
        best_cell23_mix = mix
        best_cell23_hits = hits
        best_cell23_top20 = rankings

cell23_results = (
    pd.DataFrame(rows)
    .sort_values("적중 수", ascending=False, kind="stable")
    .reset_index(drop=True)
)
print(cell23_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 EASE 비중 {best_cell23_mix:.0%} | 최고 적중 {best_cell23_hits:,}건")

output_dir = Path("/kaggle/working")
cell23_results.to_csv(output_dir / "ease_rerank_results_cell23.csv", index=False)
best_cell23_top20.to_csv(output_dir / "best_ensemble_cell23.csv", index=False)

In [ ]:
#셀 24. EASE 점수 비중 20% 이상 탐색
from pathlib import Path

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

keys = [USER_COL, ITEM_COL]
mixes = (0.20, 0.225, 0.25, 0.30, 0.35)
baseline = best_cell23_top20
_, baseline_hits = pooled_recall_at_20(baseline)

if best_cell23_mix != 0.20 or baseline_hits != 6_432:
    raise ValueError(f"셀 23 최고 설정을 확인하세요. 현재 적중: {baseline_hits:,}건")

baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
score_col = feature_names.index("cell17_score")
top_by_mix24 = {
    mix: np.empty((X.shape[0], 20), dtype=np.int32) for mix in mixes
}

with threadpool_limits(limits=4, user_api="blas"):
    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        block_scores = ease_X[start:stop] @ ease_weights

        for user in range(start, stop):
            begin, end = offsets[user:user + 2]
            items = candidate_items[begin:end]
            base_scores = features[begin:end, score_col]

            local_cols = ease_col[items]
            covered = local_cols >= 0
            ease_scores = np.zeros(len(items), dtype=np.float64)
            if covered.any():
                ease_scores[covered] = minmax(
                    block_scores[user - start, local_cols[covered]]
                )

            for mix in mixes:
                combined = (1.0 - mix) * base_scores + mix * ease_scores
                order = np.lexsort((items, -combined))[:20]
                top_by_mix24[mix][user] = items[order]

rows = []
best_cell24_mix = 0.20
best_cell24_hits = baseline_hits
best_cell24_top20 = baseline

for mix, top_items in top_by_mix24.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    if mix == 0.20 and (
        hits != baseline_hits or added != 0 or len(pairs) != len(baseline_pairs)
    ):
        raise ValueError(
            f"EASE 20% 대조군 불일치 | 적중 {hits:,}/{baseline_hits:,}건 | "
            f"신규 편입 {added:,}건"
        )

    rows.append({
        "EASE 비중": mix,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 23 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })

    if hits > best_cell24_hits:
        best_cell24_mix = mix
        best_cell24_hits = hits
        best_cell24_top20 = rankings

cell24_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell24_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 EASE 비중 {best_cell24_mix:.1%} | "
    f"최고 적중 {best_cell24_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell24_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell24_results.to_csv(output_dir / "ease_weight_results_cell24.csv", index=False)
best_cell24_top20.to_csv(output_dir / "best_ensemble_cell24.csv", index=False)

In [ ]:
#셀 25. EASE 상품 범위 확장 비교
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve
from threadpoolctl import threadpool_limits

started = perf_counter()
mix = 0.20
l2 = 300.0
score_col = feature_names.index("cell17_score")
keys = [USER_COL, ITEM_COL]

baseline = best_cell24_top20
baseline_recall, baseline_hits = pooled_recall_at_20(baseline)
if baseline_hits != 6_432 or best_cell24_mix != mix:
    raise ValueError(f"셀 24 기준선을 확인하세요. 현재 {baseline_hits:,}건")

base_pairs = pd.MultiIndex.from_frame(baseline[keys])
popularity = np.asarray(X.sum(axis=0)).ravel()
pop_order = np.lexsort((np.arange(X.shape[1]), -popularity))

def fit_expanded_ease(size):
    catalog = np.sort(pop_order[:size])
    history = X[:, catalog].tocsr().astype(np.float64)

    gram = np.asfortranarray((history.T @ history).toarray())
    gram.flat[::size + 1] += l2
    factor = cho_factor(gram, overwrite_a=True, check_finite=False)

    # 역행렬의 열을 블록 단위로 풀어 한 번에 필요한 메모리를 줄입니다.
    weights = np.empty((size, size), dtype=np.float64, order="F")
    for start in range(0, size, 256):
        stop = min(start + 256, size)
        columns = np.arange(start, stop)
        local = np.arange(stop - start)
        rhs = np.zeros((size, stop - start), dtype=np.float64, order="F")
        rhs[columns, local] = 1.0
        precision = cho_solve(
            factor, rhs, overwrite_b=True, check_finite=False
        )
        weights[:, start:stop] = -precision / precision[columns, local]
        weights[columns, columns] = 0.0

    return catalog, history, weights

def rerank_with_ease(catalog, history, weights):
    local_by_item = np.full(X.shape[1], -1, dtype=np.int32)
    local_by_item[catalog] = np.arange(len(catalog))
    top = np.empty((X.shape[0], 20), dtype=np.int32)
    covered_count = 0

    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        block_scores = history[start:stop] @ weights

        for user in range(start, stop):
            begin, end = offsets[user:user + 2]
            items = candidate_items[begin:end]
            local_cols = local_by_item[items]
            covered = local_cols >= 0
            covered_count += int(covered.sum())

            ease_scores = np.zeros(len(items), dtype=np.float64)
            if covered.any():
                ease_scores[covered] = minmax(
                    block_scores[user - start, local_cols[covered]]
                )

            combined = (1.0 - mix) * features[begin:end, score_col] + mix * ease_scores
            order = np.lexsort((items, -combined))[:20]
            top[user] = items[order]

    return make_rankings(top), covered_count

rows = []
best_cell25_top20 = baseline
best_cell25_hits = baseline_hits
best_cell25_size = len(ease_catalog)

with threadpool_limits(limits=4, user_api="blas"):
    for size in (len(ease_catalog), 5_000, 7_000):
        if size == len(ease_catalog):
            rankings = baseline
            covered_count = int(np.isin(candidate_items, ease_catalog).sum())
        else:
            if size > X.shape[1]:
                continue
            catalog, history, weights = fit_expanded_ease(size)
            rankings, covered_count = rerank_with_ease(catalog, history, weights)
            del history, weights

        recall, hits = pooled_recall_at_20(rankings)
        pairs = pd.MultiIndex.from_frame(rankings[keys])
        rows.append({
            "EASE 상품 수": size,
            "EASE 점수 적용 후보 비율": covered_count / len(candidate_items),
            "Recall@20": recall,
            "적중 수": hits,
            "셀 24 대비 적중": hits - baseline_hits,
            "Top-20 신규 편입": int((~pairs.isin(base_pairs)).sum()),
        })
        print(
            f"EASE {size:,}개 | 적중 {hits:,}건 | "
            f"누적 {perf_counter() - started:.1f}초",
            flush=True,
        )

        if hits > best_cell25_hits:
            best_cell25_size = size
            best_cell25_hits = hits
            best_cell25_top20 = rankings

cell25_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell25_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 EASE 상품 수 {best_cell25_size:,}개 | "
    f"최고 적중 {best_cell25_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell25_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell25_results.to_csv(output_dir / "ease_catalog_results_cell25.csv", index=False)
best_cell25_top20.to_csv(output_dir / "best_ensemble_cell25.csv", index=False)

In [ ]:
#셀 26. EASE 상품 범위 9,000개·11,000개 비교
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

started = perf_counter()
output_dir = Path("/kaggle/working")
keys = [USER_COL, ITEM_COL]

# 셀 25의 함수가 참조하는 설정을 명시합니다.
mix = 0.20
l2 = 300.0

baseline = best_cell25_top20
baseline_recall, baseline_hits = pooled_recall_at_20(baseline)
if best_cell25_size != 7_000 or baseline_hits != 6_471:
    raise ValueError("셀 25에서 EASE 7,000개 / 6,471건이 선택됐는지 확인하세요.")

baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
base_coverage = cell25_results.loc[
    cell25_results["EASE 상품 수"].eq(7_000), "EASE 점수 적용 후보 비율"
].iloc[0]

rows = [{
    "EASE 상품 수": 7_000,
    "EASE 점수 적용 후보 비율": base_coverage,
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "셀 25 대비 적중": 0,
    "Top-20 신규 편입": 0,
}]

best_cell26_size = 7_000
best_cell26_hits = baseline_hits
best_cell26_top20 = baseline

with threadpool_limits(limits=4, user_api="blas"):
    for size in (9_000, 11_000):
        if size > X.shape[1]:
            print(f"EASE {size:,}개 건너뜀 | 전체 상품 {X.shape[1]:,}개", flush=True)
            continue

        print(f"EASE {size:,}개 학습 시작", flush=True)
        catalog, history, weights = fit_expanded_ease(size)
        rankings, covered_count = rerank_with_ease(catalog, history, weights)
        del history, weights

        recall, hits = pooled_recall_at_20(rankings)
        pairs = pd.MultiIndex.from_frame(rankings[keys])
        rows.append({
            "EASE 상품 수": size,
            "EASE 점수 적용 후보 비율": covered_count / len(candidate_items),
            "Recall@20": recall,
            "적중 수": hits,
            "셀 25 대비 적중": hits - baseline_hits,
            "Top-20 신규 편입": int((~pairs.isin(baseline_pairs)).sum()),
        })

        if hits > best_cell26_hits:
            best_cell26_size = size
            best_cell26_hits = hits
            best_cell26_top20 = rankings

        cell26_results = pd.DataFrame(rows).sort_values(
            "적중 수", ascending=False, kind="stable"
        )
        cell26_results.to_csv(
            output_dir / "ease_catalog_results_cell26.csv", index=False
        )
        best_cell26_top20.to_csv(
            output_dir / "best_ensemble_cell26.csv", index=False
        )
        print(
            f"EASE {size:,}개 완료 | Recall@20 {recall:.6f} | "
            f"적중 {hits:,}건 ({hits - baseline_hits:+,}건) | "
            f"누적 {perf_counter() - started:.1f}초",
            flush=True,
        )

cell26_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell26_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 EASE 상품 수 {best_cell26_size:,}개 | 최고 적중 {best_cell26_hits:,}건")

In [ ]:
#셀 27. EASE L2 정규화 강도 비교
from pathlib import Path
from time import perf_counter

import pandas as pd
from threadpoolctl import threadpool_limits

started = perf_counter()
output_dir = Path("/kaggle/working")

baseline = best_cell26_top20
baseline_recall, baseline_hits = pooled_recall_at_20(baseline)
if best_cell26_size != 11_000 or baseline_hits != 6_481:
    raise ValueError("셀 26의 EASE 11,000개 / 6,481건 결과를 확인하세요.")

mix = 0.20  # 셀 25의 재정렬 함수에서 참조합니다.
rows = [{
    "EASE L2": 300,
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "셀 26 대비 적중": 0,
}]
best_cell27_l2 = 300
best_cell27_hits = baseline_hits
best_cell27_top20 = baseline

try:
    with threadpool_limits(limits=4, user_api="blas"):
        for l2 in (100.0, 900.0):  # 셀 25의 EASE 학습 함수에서 참조합니다.
            print(f"EASE L2={l2:g} 학습 시작", flush=True)
            catalog, history, ease_weights_trial = fit_expanded_ease(11_000)
            rankings, _ = rerank_with_ease(catalog, history, ease_weights_trial)
            del history, ease_weights_trial

            recall, hits = pooled_recall_at_20(rankings)
            rows.append({
                "EASE L2": int(l2),
                "Recall@20": recall,
                "적중 수": hits,
                "셀 26 대비 적중": hits - baseline_hits,
            })

            if hits > best_cell27_hits:
                best_cell27_l2 = int(l2)
                best_cell27_hits = hits
                best_cell27_top20 = rankings

            cell27_results = (
                pd.DataFrame(rows)
                .sort_values("적중 수", ascending=False, kind="stable")
                .reset_index(drop=True)
            )
            cell27_results.to_csv(output_dir / "ease_l2_results_cell27.csv", index=False)
            best_cell27_top20.to_csv(output_dir / "best_ensemble_cell27.csv", index=False)

            print(
                f"EASE L2={l2:g} | Recall@20 {recall:.6f} | "
                f"적중 {hits:,}건 ({hits - baseline_hits:+,}건) | "
                f"누적 {perf_counter() - started:.1f}초",
                flush=True,
            )
finally:
    l2 = 300.0  # 이후 셀에서 기존 EASE 설정을 참조할 수 있게 복원합니다.

print(cell27_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 L2 {best_cell27_l2} | 최고 적중 {best_cell27_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell27_hits):,}건"
)

In [ ]:
#셀 28. EASE 신규 후보를 Top-20에 직접 편입
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

started = perf_counter()
baseline = best_cell27_top20
baseline_recall, baseline_hits = pooled_recall_at_20(baseline)

if best_cell26_size != 11_000 or best_cell27_l2 != 300 or baseline_hits != 6_481:
    raise ValueError("셀 27의 기준선 11,000개 / L2=300 / 6,481건을 확인하세요.")

# 셀 16의 함수로 기준선 추천 목록을 사용자·순위 순서의 상품 열 번호로 변환합니다.
base_top = ranking_columns(baseline)
quotas = (1, 2, 3)
top_by_quota = {quota: base_top.copy() for quota in quotas}
inserted_by_quota = dict.fromkeys(quotas, 0)

# 셀 25의 EASE 학습 함수가 참조하는 정규화 강도입니다.
l2 = 300.0
print("EASE 11,000개 학습 시작", flush=True)

with threadpool_limits(limits=4, user_api="blas"):
    catalog, ease_history, ease_trial_weights = fit_expanded_ease(11_000)
    print(f"EASE 학습 완료 | {perf_counter() - started:.1f}초", flush=True)

    for start in range(0, X.shape[0], 256):
        stop = min(start + 256, X.shape[0])
        block = ease_history[start:stop]
        scores = block @ ease_trial_weights
        seen_rows, seen_cols = block.nonzero()
        scores[seen_rows, seen_cols] = -np.inf

        # 상품 열 순으로 정렬된 카탈로그에서 동점은 상품 ID 순으로 처리합니다.
        order = np.argsort(-scores, axis=1, kind="stable")[:, :50]

        for row, user in enumerate(range(start, stop)):
            begin, end = offsets[user:user + 2]
            local_top = order[row]
            local_top = local_top[np.isfinite(scores[row, local_top])]
            ease_items = catalog[local_top]
            new_items = ease_items[~np.isin(ease_items, candidate_items[begin:end])]

            for quota in quotas:
                count = min(quota, len(new_items))
                if count:
                    top_by_quota[quota][user, -count:] = new_items[:count]
                    inserted_by_quota[quota] += count

        if stop % 8192 == 0 or stop == X.shape[0]:
            print(f"후보 편입 {stop:,}/{X.shape[0]:,}명", flush=True)

del ease_history, ease_trial_weights

rows = [{
    "사용자당 최대 편입 수": 0,
    "실제 편입 건수": 0,
    "Recall@20": baseline_recall,
    "적중 수": baseline_hits,
    "셀 27 대비 적중": 0,
}]
best_cell28_quota = 0
best_cell28_hits = baseline_hits
best_cell28_top20 = baseline

for quota, top_items in top_by_quota.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    rows.append({
        "사용자당 최대 편입 수": quota,
        "실제 편입 건수": inserted_by_quota[quota],
        "Recall@20": recall,
        "적중 수": hits,
        "셀 27 대비 적중": hits - baseline_hits,
    })

    if hits > best_cell28_hits:
        best_cell28_quota = quota
        best_cell28_hits = hits
        best_cell28_top20 = rankings

cell28_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell28_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"\n선택한 최대 편입 수 {best_cell28_quota}개 | 최고 적중 {best_cell28_hits:,}건")

output_dir = Path("/kaggle/working")
cell28_results.to_csv(output_dir / "ease_new_candidates_cell28.csv", index=False)
best_cell28_top20.to_csv(output_dir / "best_ensemble_cell28.csv", index=False)

In [ ]:
#셀 29. LightGCN 후보 20·50·100개 확장 비교
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits

started = perf_counter()
depths = (20, 50, 100)
keys = [USER_COL, ITEM_COL]
model_weights = best_cell12_weights
raw_mix = best_cell17_mix
ease_mix = 0.20

baseline = best_cell28_top20
_, baseline_hits = pooled_recall_at_20(baseline)
if baseline_hits != 6_481 or best_cell26_size != 11_000 or best_cell27_l2 != 300:
    raise ValueError("셀 28까지의 기준선 EASE 11,000개 / L2=300 / 6,481건을 확인하세요.")

bm25_candidates, gcn_candidates, sx_candidates, als_candidates = candidate_lists
if tuple(a.shape[1] for a in candidate_lists) != (100, 20, 20, 100):
    raise ValueError("셀 16의 BM25·LightGCN·SimpleX·ALS 후보 목록을 확인하세요.")

top_by_depth29 = {
    depth: np.empty((X.shape[0], 20), dtype=np.int32)
    for depth in depths
}
gcn_rrf_terms = model_weights[1] * (1.0 / (20 + np.arange(1, 101)))

# 셀 25의 EASE 학습 함수가 참조하는 설정입니다.
l2 = 300.0
print("EASE 11,000개 학습 시작", flush=True)

with torch.inference_mode(), threadpool_limits(limits=4, user_api="blas"):
    catalog29, ease_history29, ease_weights29 = fit_expanded_ease(11_000)
    print(f"EASE 학습 완료 | {perf_counter() - started:.1f}초", flush=True)

    ease_local = np.full(X.shape[1], -1, dtype=np.int32)
    ease_local[catalog29] = np.arange(len(catalog29), dtype=np.int32)

    n_users29 = X.shape[0]
    gcn_vectors29 = propagate_lightgcn()
    sx_users29, sx_items29 = simplex_vectors()

    for start in range(0, n_users29, 128):
        stop = min(start + 128, n_users29)
        block = X[start:stop]
        seen_rows, seen_cols = block.nonzero()

        bm25_scores = (block @ W_item_idf).toarray()
        bm25_scores[seen_rows, seen_cols] = 0.0
        maxima = bm25_scores.max(axis=1, keepdims=True)
        np.divide(bm25_scores, maxima, out=bm25_scores, where=maxima > 0)
        bm25_scores *= 1.0 - best_pop_weight
        bm25_scores += best_pop_weight * pop_max

        gcn_scores = (
            gcn_vectors29[start:stop] @ gcn_vectors29[n_users29:].T
        ).cpu().numpy()
        gcn_scores[seen_rows, seen_cols] = -np.inf
        gcn_top100_block = np.argsort(
            -gcn_scores, axis=1, kind="stable"
        )[:, :100]

        raw_blocks = (
            bm25_scores,
            gcn_scores,
            (sx_users29[start:stop] @ sx_items29.T).cpu().numpy(),
            als_model.user_factors[start:stop] @ als_model.item_factors.T,
        )
        ease_block = ease_history29[start:stop] @ ease_weights29

        for row, user in enumerate(range(start, stop)):
            original_gcn = gcn_candidates[user]
            ranked_gcn = gcn_top100_block[row]
            tail = ranked_gcn[
                np.isfinite(gcn_scores[row, ranked_gcn])
                & ~np.isin(ranked_gcn, original_gcn)
            ]
            # 기존 1~20위는 보존하고, 새 LightGCN 후보를 뒤에 붙입니다.
            extended_gcn = np.concatenate([original_gcn, tail])

            for depth in depths:
                lists = (
                    bm25_candidates[user],
                    extended_gcn[:depth],
                    sx_candidates[user],
                    als_candidates[user],
                )
                terms = (
                    rrf_terms[0],
                    gcn_rrf_terms[:len(lists[1])],
                    rrf_terms[2],
                    rrf_terms[3],
                )
                union = np.unique(np.concatenate(lists))
                rrf_score = np.zeros(len(union), dtype=np.float64)
                raw_score = np.zeros_like(rrf_score)

                for items, rank_terms, raw, weight in zip(
                    lists, terms, raw_blocks, model_weights
                ):
                    rrf_score[np.searchsorted(union, items)] += rank_terms
                    raw_score += weight * minmax(raw[row, union].astype(np.float64))

                # 셀 18·25와 동일하게 기본 결합 점수를 float32로 사용합니다.
                base_score = (
                    (1.0 - raw_mix) * minmax(rrf_score)
                    + raw_mix * minmax(raw_score)
                ).astype(np.float32)

                local_cols = ease_local[union]
                covered = local_cols >= 0
                ease_score = np.zeros_like(rrf_score)
                if covered.any():
                    ease_score[covered] = minmax(ease_block[row, local_cols[covered]])

                final_score = (1.0 - ease_mix) * base_score + ease_mix * ease_score
                order = np.lexsort((union, -final_score))[:20]
                top_by_depth29[depth][user] = union[order]

        if stop % 8192 == 0 or stop == n_users29:
            print(f"재정렬 {stop:,}/{n_users29:,}명", flush=True)

del ease_history29, ease_weights29

baseline_pairs = pd.MultiIndex.from_frame(baseline[keys])
rows = []
best_cell29_depth = 20
best_cell29_hits = baseline_hits
best_cell29_top20 = baseline

for depth, top_items in top_by_depth29.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys])
    added = int((~pairs.isin(baseline_pairs)).sum())

    if depth == 20 and (hits != baseline_hits or added != 0):
        raise ValueError(
            f"LightGCN 20개 대조군 불일치 | 적중 {hits:,}/{baseline_hits:,}건 | "
            f"신규 편입 {added:,}건"
        )

    rows.append({
        "LightGCN 후보 수": depth,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 28 대비 적중": hits - baseline_hits,
        "Top-20 신규 편입": added,
    })

    if hits > best_cell29_hits:
        best_cell29_depth = depth
        best_cell29_hits = hits
        best_cell29_top20 = rankings

cell29_results = pd.DataFrame(rows).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell29_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 LightGCN 후보 수 {best_cell29_depth}개 | "
    f"최고 적중 {best_cell29_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell29_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell29_results.to_csv(output_dir / "lightgcn_depth_results_cell29.csv", index=False)
best_cell29_top20.to_csv(output_dir / "best_ensemble_cell29.csv", index=False)

In [ ]:
#셀 30. LightGCN 50개 고정 후 SimpleX 후보 확장 비교
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from threadpoolctl import threadpool_limits

started = perf_counter()
depths30 = (20, 50, 100)
keys30 = [USER_COL, ITEM_COL]
weights30 = best_cell12_weights
raw_mix30 = best_cell17_mix
ease_mix30 = 0.20

baseline30 = best_cell29_top20
_, baseline_hits30 = pooled_recall_at_20(baseline30)
if best_cell29_depth != 50 or baseline_hits30 != 6_488:
    raise ValueError("셀 29 최고 설정인 LightGCN 50개 / 6,488건을 확인하세요.")

baseline_pairs30 = pd.MultiIndex.from_frame(baseline30[keys30])
bm25_list30, gcn_list30, sx_list30, als_list30 = candidate_lists
if tuple(items.shape[1] for items in candidate_lists) != (100, 20, 20, 100):
    raise ValueError("셀 16의 모델별 후보 목록을 확인하세요.")

n_users30 = X.shape[0]
top_by_depth30 = {
    depth: np.empty((n_users30, 20), dtype=np.int32) for depth in depths30
}
gcn_terms30 = weights30[1] * (1.0 / (20 + np.arange(1, 51)))
sx_terms30 = weights30[2] * (1.0 / (20 + np.arange(1, 101)))

def extend_candidates30(original, ranked, scores, depth):
    tail = ranked[np.isfinite(scores[ranked]) & ~np.isin(ranked, original)]
    return np.concatenate([original, tail])[:depth]

# 셀 25의 EASE 학습 함수가 참조하는 설정입니다.
l2 = 300.0

print("EASE 11,000개 학습 시작", flush=True)
with torch.inference_mode(), threadpool_limits(limits=4, user_api="blas"):
    catalog30, history30, ease_weights30 = fit_expanded_ease(11_000)
    print(f"EASE 학습 완료 | {perf_counter() - started:.1f}초", flush=True)

    ease_local30 = np.full(X.shape[1], -1, dtype=np.int32)
    ease_local30[catalog30] = np.arange(len(catalog30), dtype=np.int32)
    gcn30 = propagate_lightgcn()
    sx_users30, sx_items30 = simplex_vectors()

    for start in range(0, n_users30, 128):
        stop = min(start + 128, n_users30)
        block = X[start:stop]
        seen_rows, seen_cols = block.nonzero()

        bm25 = (block @ W_item_idf).toarray()
        bm25[seen_rows, seen_cols] = 0.0
        maxima = bm25.max(axis=1, keepdims=True)
        np.divide(bm25, maxima, out=bm25, where=maxima > 0)
        bm25 *= 1.0 - best_pop_weight
        bm25 += best_pop_weight * pop_max

        gcn_scores = (gcn30[start:stop] @ gcn30[n_users30:].T).cpu().numpy()
        sx_scores = (sx_users30[start:stop] @ sx_items30.T).cpu().numpy()
        gcn_scores[seen_rows, seen_cols] = -np.inf
        sx_scores[seen_rows, seen_cols] = -np.inf

        gcn_ranked = np.argsort(-gcn_scores, axis=1, kind="stable")[:, :100]
        sx_ranked = np.argsort(-sx_scores, axis=1, kind="stable")[:, :100]
        als_scores = als_model.user_factors[start:stop] @ als_model.item_factors.T
        raw_blocks = (bm25, gcn_scores, sx_scores, als_scores)
        ease_block = history30[start:stop] @ ease_weights30

        for row, user in enumerate(range(start, stop)):
            gcn_candidates30 = extend_candidates30(
                gcn_list30[user], gcn_ranked[row], gcn_scores[row], 50
            )
            sx_candidates30 = extend_candidates30(
                sx_list30[user], sx_ranked[row], sx_scores[row], 100
            )

            for depth in depths30:
                lists = (
                    bm25_list30[user], gcn_candidates30,
                    sx_candidates30[:depth], als_list30[user],
                )
                terms = (
                    rrf_terms[0], gcn_terms30[:len(lists[1])],
                    sx_terms30[:len(lists[2])], rrf_terms[3],
                )
                union = np.unique(np.concatenate(lists))
                rrf_score = np.zeros(len(union), dtype=np.float64)
                raw_score = np.zeros_like(rrf_score)

                for items, rank_terms, raw, weight in zip(
                    lists, terms, raw_blocks, weights30
                ):
                    rrf_score[np.searchsorted(union, items)] += rank_terms
                    raw_score += weight * minmax(raw[row, union].astype(np.float64))

                base_score = (
                    (1.0 - raw_mix30) * minmax(rrf_score)
                    + raw_mix30 * minmax(raw_score)
                ).astype(np.float32)

                local_cols = ease_local30[union]
                covered = local_cols >= 0
                ease_score = np.zeros_like(rrf_score)
                if covered.any():
                    ease_score[covered] = minmax(ease_block[row, local_cols[covered]])

                final_score = (1.0 - ease_mix30) * base_score + ease_mix30 * ease_score
                top_by_depth30[depth][user] = union[
                    np.lexsort((union, -final_score))[:20]
                ]

        if stop % 8192 == 0 or stop == n_users30:
            print(f"재정렬 {stop:,}/{n_users30:,}명", flush=True)

del history30, ease_weights30

rows30 = []
best_cell30_depth = 20
best_cell30_hits = baseline_hits30
best_cell30_top20 = baseline30

for depth, top_items in top_by_depth30.items():
    rankings = make_rankings(top_items)
    recall, hits = pooled_recall_at_20(rankings)
    pairs = pd.MultiIndex.from_frame(rankings[keys30])
    added = int((~pairs.isin(baseline_pairs30)).sum())

    if depth == 20 and (hits != baseline_hits30 or added != 0):
        raise ValueError(
            f"SimpleX 20개 대조군 불일치 | 적중 {hits:,}/{baseline_hits30:,}건 | "
            f"신규 편입 {added:,}건"
        )

    rows30.append({
        "SimpleX 후보 수": depth,
        "Recall@20": recall,
        "적중 수": hits,
        "셀 29 대비 적중": hits - baseline_hits30,
        "Top-20 신규 편입": added,
    })

    if hits > best_cell30_hits:
        best_cell30_depth = depth
        best_cell30_hits = hits
        best_cell30_top20 = rankings

cell30_results = pd.DataFrame(rows30).sort_values(
    "적중 수", ascending=False, kind="stable"
)
print(cell30_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

target_hits = (len(truth_pairs) + 19) // 20
print(
    f"\n선택한 SimpleX 후보 수 {best_cell30_depth}개 | "
    f"최고 적중 {best_cell30_hits:,}건 | "
    f"0.05 목표까지 {max(0, target_hits - best_cell30_hits):,}건"
)

output_dir = Path("/kaggle/working")
cell30_results.to_csv(output_dir / "simplex_depth_results_cell30.csv", index=False)
best_cell30_top20.to_csv(output_dir / "best_ensemble_cell30.csv", index=False)

In [ ]:
#셀 31. 전체 학습·테스트·제출 예시 파일 형식 확인
from pathlib import Path

import pandas as pd

competition_dir = (
    Path("/kaggle/input/competitions")
    / "ssu-data-science-amazon-top-k-recommendation"
)

for filename in ("train.csv", "test.csv", "sample_submission.csv"):
    path = competition_dir / filename
    preview = pd.read_csv(path, dtype=str, nrows=2)
    print(f"\n{filename} | 열 이름 {preview.columns.tolist()}")
    print(preview.to_string(index=False))

In [ ]:
#셀 32. 전체 train.csv로 제출용 모델 재학습
from pathlib import Path
from types import FunctionType

import numpy as np
import pandas as pd
from scipy import sparse
from threadpoolctl import threadpool_limits

required32 = (
    "blocks", "best_cell30_depth", "best_cell29_depth",
    "best_cell12_weights", "best_cell17_mix", "best_pop_weight",
    "make_rankings", "K1", "B", "POWER", "SHRINK", "NEIGHBORS",
)
missing32 = [name for name in required32 if name not in globals()]
if missing32:
    raise RuntimeError(f"이전 실험 셀의 실행 결과가 필요해요: {', '.join(missing32)}")
if (best_cell29_depth, best_cell30_depth) != (50, 100):
    raise ValueError("셀 29·30에서 선택한 LightGCN 50개 / SimpleX 100개를 확인하세요.")

data_dir32 = (
    Path("/kaggle/input/competitions")
    / "ssu-data-science-amazon-top-k-recommendation"
)
full_train32 = pd.read_csv(
    data_dir32 / "train.csv", usecols=["user_id", "item_id"], dtype=str
)
test32 = pd.read_csv(data_dir32 / "test.csv", dtype=str)
sample32 = pd.read_csv(data_dir32 / "sample_submission.csv", dtype=str)

for name, frame in (
    ("train", full_train32), ("test", test32), ("sample_submission", sample32)
):
    if frame.isna().any().any():
        raise ValueError(f"{name}에 빈 값이 있어요.")

test_ids32 = pd.Index(test32["user_id"])
sample_ids32 = pd.Index(sample32["user_id"])
if (
    not test_ids32.is_unique
    or not sample_ids32.is_unique
    or set(test_ids32) != set(sample_ids32)
):
    raise ValueError("test.csv와 sample_submission.csv의 사용자 목록을 확인하세요.")

# 검증 때와 동일하게 상품 ID를 문자열로 정렬해 동점을 처리합니다.
train_ids32 = pd.Index(full_train32["user_id"].unique())
all_users32 = test_ids32.append(train_ids32[~train_ids32.isin(test_ids32)])
all_items32 = pd.Index(sorted(full_train32["item_id"].unique()))

user_rows32 = all_users32.get_indexer(full_train32["user_id"])
item_cols32 = all_items32.get_indexer(full_train32["item_id"])
full_X32 = sparse.csr_matrix(
    (np.ones(len(full_train32), dtype=np.float32), (user_rows32, item_cols32)),
    shape=(len(all_users32), len(all_items32)),
)
full_X32.data.fill(1.0)
full_X32.sort_indices()

print(
    f"전체 학습 {full_X32.nnz:,}건 | 학습 행렬 {full_X32.shape} | "
    f"테스트 사용자 {len(test_ids32):,}명",
    flush=True,
)

submit_ns = {
    "X": full_X32,
    "eval_user_ids": all_users32,
    "item_ids": all_items32,
    "USER_COL": "user_id",
    "ITEM_COL": "item_id",
    "TOP_K": 20,
    "np": np,
    "pd": pd,
    "sparse": sparse,
    "threadpool_limits": threadpool_limits,
    "K1": K1,
    "B": B,
    "POWER": POWER,
    "SHRINK": SHRINK,
    "NEIGHBORS": NEIGHBORS,
    "best_pop_weight": best_pop_weight,
    "best_cell12_weights": best_cell12_weights,
    "best_cell17_mix": best_cell17_mix,
    "l2": 300.0,
}
submit_ns["make_rankings"] = FunctionType(make_rankings.__code__, submit_ns)

# 셀 6·9의 학습 전 점검에서 이름을 확인합니다.
# 아래에서 재사용하는 학습·추천 코드에는 검증 채점 호출이 없습니다.
submit_ns["pooled_recall_at_20"] = None

for name in ("BM25 함수", "후보 확장 함수", "후보 목록 함수"):
    for source in blocks[name]:
        exec(source, submit_ns)

print("제출용 BM25-Shrink 학습", flush=True)
lengths32 = np.diff(full_X32.indptr).astype(np.float32)
pop32 = np.asarray(full_X32.sum(axis=0)).ravel()
idf32 = np.log1p(
    (full_X32.shape[0] - pop32 + 0.5) / (pop32 + 0.5)
).astype(np.float32)
norm32 = K1 * (1.0 - B + B * lengths32 / lengths32.mean())
weighted32 = full_X32.copy()
weighted32.data = (
    idf32[weighted32.indices]
    * np.repeat((K1 + 1.0) / (1.0 + norm32), np.diff(weighted32.indptr))
).astype(np.float32)

submit_ns["W_item_idf"] = submit_ns["build_item_neighbors"](weighted32.T.tocsr())
submit_ns["pop_max"] = pop32 / pop32.max()
submit_ns["max_depth"] = 100
submit_ns["bm25_top100"] = submit_ns["expanded_top_items"]("BM25")
del weighted32

# 셀 6이 요구하는 변수 이름을 채웁니다.
# 실제 최종 결합에는 아래에서 생성하는 모델별 후보를 사용합니다.
submit_ns["bm25_lightgcn_val_top20"] = submit_ns["make_rankings"](
    submit_ns["bm25_top100"][:, :20]
)

print("제출용 LightGCN 학습", flush=True)
exec(blocks["LightGCN"][0], submit_ns)
submit_ns["lightgcn_val_top20"] = submit_ns["lightgcn_top20"]()

print("제출용 SimpleX 학습", flush=True)
exec(blocks["SimpleX"][0], submit_ns)

print("제출용 ALS 학습", flush=True)
exec(blocks["ALS"][0], submit_ns)
submit_ns["als_top100"] = submit_ns["expanded_top_items"]("ALS")

print("네 기본 모델 학습 완료. 셀 33을 실행하세요.", flush=True)

In [ ]:
#셀 33. 최종 앙상블 추천 생성과 submission.csv 저장
from IPython import get_ipython
from pathlib import Path
from time import perf_counter
from types import FunctionType

import numpy as np
import pandas as pd
import torch
from scipy.linalg import cho_factor, cho_solve
from threadpoolctl import threadpool_limits

if "submit_ns" not in globals() or "als_top100" not in submit_ns:
    raise RuntimeError("셀 32를 끝까지 실행하세요.")

# 셀 30에서 검증한 재정렬 루프를 재사용합니다.
title33 = "#셀 30. LightGCN 50개 고정 후 SimpleX 후보 확장 비교"
source30 = next(
    (
        source for source in reversed(get_ipython().user_ns["In"])
        if source.strip().splitlines()
        and source.strip().splitlines()[0].strip() == title33
    ),
    None,
)
if source30 is None:
    raise RuntimeError("현재 세션의 실행 기록에서 셀 30을 찾지 못했어요.")

start_marker33 = 'print("EASE 11,000개 학습 시작", flush=True)'
end_marker33 = "del history30, ease_weights30"
start33 = source30.find(start_marker33)
end33 = source30.find(end_marker33, start33)
if start33 < 0 or end33 < 0:
    raise RuntimeError("셀 30의 EASE 학습·재정렬 구간을 찾지 못했어요.")

rerank_code33 = compile(
    source30[start33:end33 + len(end_marker33)], "<제출용 재정렬>", "exec"
)

n_users33, n_items33 = submit_ns["X"].shape
weights33 = submit_ns["best_cell12_weights"]
gcn_candidates33 = submit_ns["ranking_columns"](submit_ns["lightgcn_val_top20"])
sx_candidates33 = submit_ns["ranking_columns"](submit_ns["simplex_val_top20"])

submit_ns.update({
    "torch": torch,
    "perf_counter": perf_counter,
    "cho_factor": cho_factor,
    "cho_solve": cho_solve,
    "started": perf_counter(),
    "n_users30": n_users33,
    "depths30": (100,),
    "top_by_depth30": {100: np.empty((n_users33, 20), dtype=np.int32)},
    "weights30": weights33,
    "raw_mix30": submit_ns["best_cell17_mix"],
    "ease_mix30": 0.20,
    "bm25_list30": submit_ns["bm25_top100"],
    "gcn_list30": gcn_candidates33,
    "sx_list30": sx_candidates33,
    "als_list30": submit_ns["als_top100"],
    "gcn_terms30": weights33[1] * (1.0 / (20 + np.arange(1, 51))),
    "sx_terms30": weights33[2] * (1.0 / (20 + np.arange(1, 101))),
    "rrf_terms": tuple(
        weight * (1.0 / (20 + np.arange(1, depth + 1)))
        for weight, depth in zip(weights33, (100, 20, 20, 100))
    ),
    "l2": 300.0,
})
pop33 = np.asarray(submit_ns["X"].sum(axis=0)).ravel()
submit_ns["pop_order"] = np.lexsort((np.arange(n_items33), -pop33))
submit_ns["fit_expanded_ease"] = FunctionType(fit_expanded_ease.__code__, submit_ns)
submit_ns["extend_candidates30"] = FunctionType(extend_candidates30.__code__, submit_ns)

if n_items33 < 11_000:
    raise ValueError(f"EASE 학습에 상품 11,000개가 필요해요. 현재 {n_items33:,}개")

exec(rerank_code33, submit_ns)

# sample_submission.csv의 사용자 순서를 그대로 따릅니다.
sample_rows33 = all_users32.get_indexer(sample_ids32)
top33 = submit_ns["top_by_depth30"][100][sample_rows33].copy()

# 학습 이력이 없는 테스트 사용자는 전체 인기도 기준으로 추천합니다.
cold33 = np.diff(full_X32.indptr)[sample_rows33] == 0
top33[cold33] = submit_ns["pop_order"][:20]

if top33.shape != (len(sample32), 20):
    raise ValueError("사용자별 추천 개수가 20개인지 확인하세요.")
if ((top33 < 0) | (top33 >= n_items33)).any():
    raise ValueError("상품 범위를 벗어난 추천이 있어요.")
if (np.diff(np.sort(top33, axis=1), axis=1) == 0).any():
    raise ValueError("같은 사용자에게 중복 추천된 상품이 있어요.")
if np.asarray(full_X32[np.repeat(sample_rows33, 20), top33.ravel()]).any():
    raise ValueError("전체 학습 데이터에서 이미 본 상품이 추천에 포함됐어요.")

item_values33 = all_items32.to_numpy()
submission = pd.DataFrame({
    "user_id": sample_ids32.to_numpy(),
    "item_id": [" ".join(item_values33[row]) for row in top33],
})
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)

print(f"제출 파일 저장 완료 | {len(submission):,}명 × 20개")
print(f"학습 이력이 없어 인기도를 적용한 사용자: {int(cold33.sum()):,}명")
print(submission_path)
display(submission.head(2))

In [ ]:
#셀 34. 검증용 Top-20 추천 CSV를 ZIP으로 저장
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd

work_dir34 = Path("/kaggle/working")
zip_path34 = work_dir34 / "validation_top20_experiments.zip"
temp_path34 = work_dir34 / "validation_top20_experiments.zip.tmp"

columns34 = [USER_COL, ITEM_COL, "rank"]
expected_rows34 = len(eval_user_ids) * 20
manifest34 = []

# 개별 모델과 셀별 최고 추천 목록 중 현재 세션에 남아 있는 결과
frame_names34 = (
    "hyerin_val_top20",
    "bm25_debug_val_top20",
    "lightgcn_val_top20",
    "bm25_lightgcn_val_top20",
    "simplex_val_top20",
    "next_ensemble_val_top20",
    "best_after_direct_fusion",
    "als_val_top20",
    "best_after_als_val_top20",
    "best_cell11_top20",
    "best_cell12_top20",
    "best_cell13_top20",
    "best_cell15_top20",
    "best_cell16_top20",
    "best_cell17_top20",
    "inner_blended_val_top20",
    "ease_val_top20",
    *[f"best_cell{cell}_top20" for cell in range(23, 31)],
)

# 설정별 Top-20 상품 배열이 남아 있는 비교 실험
sweeps34 = {
    "cell04_popularity_weight": "top_by_weight",
    "cell16_raw_score_mix": "top_by_mix",
    "cell17_raw_score_mix": "cell17_top_by_mix",
    "cell23_ease_weight": "ease_top_by_mix",
    "cell24_ease_weight": "top_by_mix24",
    "cell28_ease_quota": "top_by_quota",
    "cell29_lightgcn_depth": "top_by_depth29",
    "cell30_simplex_depth": "top_by_depth30",
}

def write_top20_csv34(archive, filename, frame, source):
    if not set(columns34).issubset(frame.columns):
        raise ValueError(f"{filename}: 검증 추천 CSV 열을 확인하세요.")
    if len(frame) != expected_rows34:
        raise ValueError(
            f"{filename}: {len(frame):,}행 / 예상 {expected_rows34:,}행"
        )

    with archive.open(filename, "w", force_zip64=True) as output:
        frame.to_csv(output, columns=columns34, index=False, encoding="utf-8")

    manifest34.append({
        "file": filename,
        "source": source,
        "rows": len(frame),
    })

with ZipFile(temp_path34, "w", compression=ZIP_DEFLATED, compresslevel=3) as archive:
    # 이미 저장한 검증용 추천 CSV만 추가합니다.
    for path in sorted(work_dir34.glob("*.csv")):
        is_validation_top20 = (
            path.name.endswith("_val_top20.csv")
            or path.name.startswith("best_ensemble_")
        )
        if is_validation_top20:
            frame = pd.read_csv(path, dtype={USER_COL: str, ITEM_COL: str})
            write_top20_csv34(archive, f"saved_csv/{path.name}", frame, path.name)

    for name in frame_names34:
        frame = globals().get(name)
        if isinstance(frame, pd.DataFrame):
            write_top20_csv34(
                archive, f"session_best/{name}.csv", frame, name
            )

    for experiment, variable in sweeps34.items():
        settings = globals().get(variable)
        if not isinstance(settings, dict):
            continue

        for setting, top_items in sorted(settings.items()):
            if not isinstance(top_items, np.ndarray):
                continue
            if top_items.shape != (len(eval_user_ids), 20):
                raise ValueError(f"{variable}[{setting}]: Top-20 배열 크기를 확인하세요.")

            rankings = make_rankings(top_items)
            write_top20_csv34(
                archive,
                f"settings/{experiment}_{setting:g}.csv",
                rankings,
                f"{variable}[{setting}]",
            )

    if not manifest34:
        raise RuntimeError("현재 세션에서 검증용 Top-20 결과를 찾지 못했어요.")

    archive.writestr("manifest.csv", pd.DataFrame(manifest34).to_csv(index=False))

# ZIP을 다시 열어 모든 항목을 읽을 수 있는지 검사합니다.
with ZipFile(temp_path34) as archive:
    bad_file34 = archive.testzip()
    if bad_file34 is not None:
        raise RuntimeError(f"ZIP 검사 실패: {bad_file34}")

temp_path34.replace(zip_path34)

print(f"검증 추천 CSV {len(manifest34):,}개 저장 완료")
print(f"ZIP 크기: {zip_path34.stat().st_size / 1024**2:.1f} MB")
print(zip_path34)